# World Athletics Singapore — Incremental, Audited & Versioned Production Extraction (v3.1)

This notebook is the incremental successor to the optimized production extractor.

## Default production behaviour

A normal run **does not rebuild the 2026 dataset from scratch**. It:

1. loads the existing current production tables;
2. refreshes only athlete profiles that are due for a result check;
3. compares the newly returned World Athletics data with the current stored data;
4. applies only inserts / updates / deletions inside a configurable recent reconciliation window;
5. processes only newly encountered competitions;
6. refreshes enrichment only for athletes whose results changed, newly discovered athletes, missing enrichment, or stale enrichment;
7. atomically updates the current production files;
8. writes immutable version snapshots for every table that changed;
9. writes row-level result deltas and table-level audit records;
10. records code version, semantic notebook code hash, run ID, data hashes, API statistics, source exceptions and watermarks.

### Important World Athletics API constraint

The proven athlete-results API accepts `athlete + year`, not an `updated_since` timestamp. Therefore the API can still return the selected year's result history for an athlete that is refreshed. The pipeline is incremental in two ways:

- **only athletes due for refresh are queried**; and
- **only changed rows in the configured reconciliation window are merged/persisted**.

Set `FORCE_FULL_YEAR_RECONCILIATION=True` periodically (for example monthly) to detect old backfills/corrections outside the normal lookback window.

### Current vs history

- Root-level Parquet/CSV files are the **current canonical snapshot**.
- `deltas/<RUN_ID>/` is append-only row-level change evidence for that run.
- `versions/<RUN_ID>/tables/` contains immutable snapshots of every table changed by that run.
- `audit/` contains cumulative run, code-version, table-version and result-change history.

**v3.1:** every athlete-keyed extract contains `athleteName` directly after `aaAthleteId`.

In [31]:
# 1. Imports, pipeline version and production configuration

from __future__ import annotations

import asyncio
import hashlib
import json
import logging
import os
import random
import re
import time
import unicodedata
import uuid

from collections import deque
from datetime import datetime, timezone, timedelta
from pathlib import Path
from typing import Any, Awaitable, Callable, Iterable

import httpx
import pandas as pd
from playwright.async_api import async_playwright


# ---------------------------------------------------------------------
# PIPELINE / CODE VERSIONING
# ---------------------------------------------------------------------

PIPELINE_NAME = "world_athletics_sgp_incremental"
PIPELINE_VERSION = "3.1.0"

# IMPORTANT:
# Every intentional code/logic change must:
#   1) bump PIPELINE_VERSION; and
#   2) add a short entry here.
CODE_CHANGELOG = [
    {
        "version": "3.1.0",
        "date": "2026-09-21",
        "change": (
            "Added athleteName to every athlete-keyed extract, audit table and delta. "
            "Names are joined from athlete_master and excluded from result hashes."
        ),
    },
    {
        "version": "3.0.0",
        "date": "2026-09-20",
        "change": (
            "Introduced incremental result refresh, recent-window reconciliation, "
            "delta append history, table snapshots, code-hash guard, per-athlete "
            "refresh state, new-competition-only closure, targeted enrichment refresh, "
            "and comprehensive run/table/result audit logging."
        ),
    }
]

ENFORCE_CODE_VERSION_BUMP = True
NOTEBOOK_FILENAME = "world_athletics_SGP_2026_incremental_versioned.ipynb"


# ---------------------------------------------------------------------
# EXTRACTION SCOPE
# ---------------------------------------------------------------------

COUNTRY_CODE = "SGP"

# Override with WA_RESULTS_YEAR if needed, e.g.:
# export WA_RESULTS_YEAR=2026
RESULTS_YEAR = int(
    os.getenv(
        "WA_RESULTS_YEAR",
        str(datetime.now(timezone.utc).year)
    )
)

OUTPUT_DIR = Path(f"world_athletics_{COUNTRY_CODE}_{RESULTS_YEAR}")

RAW_DIR = OUTPUT_DIR / "raw"
RAW_RESULTS_DIR = RAW_DIR / "results" / str(RESULTS_YEAR)
RAW_COMPETITION_DIR = RAW_DIR / "competitions"
RAW_ENRICHMENT_DIR = RAW_DIR / "enrichment"

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
STATE_DIR = OUTPUT_DIR / "state"
AUDIT_DIR = OUTPUT_DIR / "audit"
VERSIONS_DIR = OUTPUT_DIR / "versions"
DELTAS_DIR = OUTPUT_DIR / "deltas"
LOG_DIR = OUTPUT_DIR / "logs"

RUN_STARTED_AT = datetime.now(timezone.utc)
RUN_ID = (
    RUN_STARTED_AT.strftime("%Y%m%dT%H%M%SZ")
    + "_"
    + uuid.uuid4().hex[:8]
)
DATA_VERSION = RUN_ID

RUN_VERSION_DIR = VERSIONS_DIR / RUN_ID
RUN_DELTA_DIR = DELTAS_DIR / RUN_ID

for p in [
    OUTPUT_DIR,
    RAW_DIR,
    RAW_RESULTS_DIR,
    RAW_COMPETITION_DIR,
    RAW_ENRICHMENT_DIR,
    CHECKPOINT_DIR,
    STATE_DIR,
    AUDIT_DIR,
    VERSIONS_DIR,
    DELTAS_DIR,
    LOG_DIR,
    RUN_VERSION_DIR / "tables",
    RUN_DELTA_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# INCREMENTAL REFRESH POLICY
# ---------------------------------------------------------------------

USE_EXISTING_MASTER_AS_SEED = True

# Results:
# Active athletes are checked on every normal manual production run.
ACTIVE_RESULT_REFRESH_HOURS = 0

# Profiles with no current-year results are checked less frequently.
INACTIVE_RESULT_REFRESH_DAYS = 7

# Reconcile only the recent part of an athlete's season during normal runs.
# The WA endpoint returns the year; this limits what is compared/merged.
RESULT_RECONCILIATION_LOOKBACK_DAYS = 45

# Turn True periodically to detect older backfills/corrections/deletions.
FORCE_FULL_YEAR_RECONCILIATION = False

# Enrichment is refreshed when results changed/new athlete, plus periodically.
ENRICHMENT_REFRESH_DAYS = 7

# The validated athlete master has already undergone full surname/token closure.
# Enable periodically for a deliberate discovery completeness refresh.
RUN_SEARCH_INDEX_REFRESH = False
SEARCH_INCREMENTAL_ONLY = True

# Competition handling.
RETRY_KNOWN_COMPETITION_EXCEPTIONS = False
KNOWN_COMPETITION_EXCEPTIONS = {7243069}


# ---------------------------------------------------------------------
# SAFE CONCURRENCY
# ---------------------------------------------------------------------

API_MAX_CONCURRENCY = 8
SEARCH_QUERY_CONCURRENCY = 4
PROFILE_VALIDATION_CONCURRENCY = 6
RESULT_FETCH_CONCURRENCY = 8
COMPETITION_FETCH_CONCURRENCY = 5
ENRICHMENT_FETCH_CONCURRENCY = 6

HTTP_TIMEOUT_SECONDS = 60.0
HTTP_MAX_RETRIES = 4

SEARCH_INDEX_MAX_PAGES = 10
SEARCH_INDEX_STALE_PAGE_LIMIT = 2

EXCLUDE_SEARCH_TOKENS = {
    "BIN", "BINTI", "BINTE", "MOHD", "MUHD",
    "MUHAMMAD", "ABDUL", "AHMAD"
}


# ---------------------------------------------------------------------
# LOGGING
# ---------------------------------------------------------------------

logger = logging.getLogger("wa_incremental_versioned")
logger.setLevel(logging.INFO)

if not logger.handlers:
    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )

    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)

    file_handler = logging.FileHandler(
        LOG_DIR / f"run_{RUN_ID}.log",
        encoding="utf-8"
    )
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)


print("Pipeline:", PIPELINE_NAME)
print("Pipeline version:", PIPELINE_VERSION)
print("Run ID:", RUN_ID)
print("Results year:", RESULTS_YEAR)
print("Output directory:", OUTPUT_DIR.resolve())
print("Search-index refresh:", RUN_SEARCH_INDEX_REFRESH)
print("Full-year reconciliation:", FORCE_FULL_YEAR_RECONCILIATION)

Pipeline: world_athletics_sgp_incremental
Pipeline version: 3.1.0
Run ID: 20260927T071746Z_b5c61458
Results year: 2026
Output directory: /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_SGP_2026
Search-index refresh: False
Full-year reconciliation: False


In [32]:
import sys
import subprocess

print(sys.executable)

subprocess.run(
    [
        sys.executable,
        "-m",
        "playwright",
        "install",
        "chromium"
    ],
    check=True
)

/Users/veesheenyuen/.pyenv/versions/SAA/bin/python3.10


CompletedProcess(args=['/Users/veesheenyuen/.pyenv/versions/SAA/bin/python3.10', '-m', 'playwright', 'install', 'chromium'], returncode=0)

In [33]:
# 2. Progress, atomic I/O, hashing, append-only audit helpers

def format_duration(seconds: float | None) -> str:
    if seconds is None or seconds < 0:
        return "?"
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h:
        return f"{h}h {m:02d}m {s:02d}s"
    if m:
        return f"{m}m {s:02d}s"
    return f"{s}s"


def print_progress(
    label: str,
    done: int,
    total: int | None,
    started: float,
    *,
    extra: str = ""
) -> None:
    elapsed = max(time.monotonic() - started, 1e-9)
    rate = done / elapsed if done else 0.0

    if total is not None and total > 0:
        remaining = max(total - done, 0)
        eta = remaining / rate if rate > 0 else None
        base = (
            f"{label}: {done:,}/{total:,} "
            f"({done/total:.1%}) | elapsed={format_duration(elapsed)} "
            f"| rate={rate:.2f}/s | ETA={format_duration(eta)}"
        )
    else:
        base = (
            f"{label}: {done:,} | elapsed={format_duration(elapsed)} "
            f"| rate={rate:.2f}/s"
        )

    if extra:
        base += f" | {extra}"

    print(base, flush=True)


def write_json_atomic(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(
            obj,
            f,
            ensure_ascii=False,
            indent=2,
            default=str
        )
    os.replace(tmp, path)


def read_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_parquet_atomic(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    df.to_parquet(tmp, index=False)
    os.replace(tmp, path)


def save_csv_atomic(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def load_current_table(name: str) -> pd.DataFrame:
    path = OUTPUT_DIR / f"{name}.parquet"
    if not path.exists():
        return pd.DataFrame()
    return pd.read_parquet(path)


def safe_int_set(values: Iterable[Any]) -> set[int]:
    values = list(values)
    if not values:
        return set()

    return set(
        pd.to_numeric(
            pd.Series(values),
            errors="coerce"
        )
        .dropna()
        .astype(int)
        .tolist()
    )


def canonical_value(value: Any) -> Any:
    if value is None:
        return None

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    if isinstance(value, (list, tuple)):
        return [canonical_value(x) for x in value]

    if isinstance(value, set):
        return sorted(canonical_value(x) for x in value)

    if isinstance(value, dict):
        return {
            str(k): canonical_value(v)
            for k, v in sorted(
                value.items(),
                key=lambda kv: str(kv[0])
            )
        }

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    # Convert numpy scalar-like values cleanly.
    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            pass

    return value


def stable_record_hash(
    row: pd.Series | dict,
    columns: list[str]
) -> str:
    payload = {
        col: canonical_value(
            row.get(col)
            if hasattr(row, "get")
            else None
        )
        for col in columns
    }

    encoded = json.dumps(
        payload,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str
    ).encode("utf-8")

    return hashlib.sha256(encoded).hexdigest()


def make_hashable(value: Any) -> Any:
    if isinstance(value, (list, dict, tuple, set)):
        return json.dumps(
            canonical_value(value),
            sort_keys=True,
            ensure_ascii=False,
            default=str
        )
    return value


def json_safe_export_copy(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in out.columns:
        if out[col].dtype != "object":
            continue

        has_nested = out[col].map(
            lambda x: isinstance(
                x,
                (list, dict, tuple, set)
            )
        ).any()

        if has_nested:
            out[col] = out[col].map(
                lambda x: make_hashable(x)
                if isinstance(
                    x,
                    (list, dict, tuple, set)
                )
                else x
            )

    return out


def dataframe_sha256(df: pd.DataFrame) -> str:
    """
    Order-insensitive content hash for audit/versioning.
    """
    if df is None:
        return hashlib.sha256(b"<NONE>").hexdigest()

    if df.empty:
        payload = json.dumps(
            {
                "columns": list(df.columns),
                "dtypes": {
                    c: str(t)
                    for c, t in df.dtypes.items()
                },
                "rows": 0
            },
            sort_keys=True
        )
        return hashlib.sha256(
            payload.encode("utf-8")
        ).hexdigest()

    safe = json_safe_export_copy(df)

    # Stable column order for hashing only.
    safe = safe.reindex(
        sorted(safe.columns),
        axis=1
    )

    # Convert to strings to make mixed/object types deterministic.
    text = safe.copy()
    for col in text.columns:
        text[col] = text[col].map(
            lambda x: "<NA>"
            if x is None or (
                not isinstance(x, (list, dict, tuple, set))
                and pd.isna(x)
            )
            else str(x)
        )

    row_hashes = pd.util.hash_pandas_object(
        text,
        index=False
    ).sort_values()

    h = hashlib.sha256()
    h.update(
        json.dumps(
            list(text.columns),
            separators=(",", ":")
        ).encode("utf-8")
    )
    h.update(row_hashes.values.tobytes())
    return h.hexdigest()


def semantic_notebook_code_hash(path: Path) -> str | None:
    """
    Hash notebook cell sources only.
    Outputs/execution counts are excluded so rerunning the notebook
    does not look like a code change.
    """
    if not path.exists():
        return None

    try:
        import nbformat

        nb = nbformat.read(path, as_version=4)
        source_blob = "\n\n".join(
            f"{cell.cell_type}\n{cell.source}"
            for cell in nb.cells
        )

        return hashlib.sha256(
            source_blob.encode("utf-8")
        ).hexdigest()

    except Exception as exc:
        logger.warning(
            "Could not compute semantic notebook code hash: %r",
            exc
        )
        return None


def append_audit_parquet(
    path: Path,
    rows: pd.DataFrame
) -> pd.DataFrame:
    if rows is None or rows.empty:
        return (
            pd.read_parquet(path)
            if path.exists()
            else pd.DataFrame()
        )

    existing = (
        pd.read_parquet(path)
        if path.exists()
        else pd.DataFrame()
    )

    combined = pd.concat(
        [existing, rows],
        ignore_index=True,
        sort=False
    )

    save_parquet_atomic(
        combined,
        path
    )

    return combined


def table_schema_hash(df: pd.DataFrame) -> str:
    payload = json.dumps(
        {
            col: str(dtype)
            for col, dtype in df.dtypes.items()
        },
        sort_keys=True
    )
    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()

In [34]:
# 3. Version guard: enforce a pipeline-version bump when notebook code changes

PIPELINE_STATE_PATH = STATE_DIR / "pipeline_state.json"

prior_pipeline_state = (
    read_json(PIPELINE_STATE_PATH)
    if PIPELINE_STATE_PATH.exists()
    else {}
)

# Try the expected filename first, then a matching file in the cwd.
notebook_path = Path.cwd() / NOTEBOOK_FILENAME

if not notebook_path.exists():
    candidates = sorted(
        Path.cwd().glob(
            "world_athletics_SGP_2026_incremental_versioned*.ipynb"
        )
    )
    if candidates:
        notebook_path = candidates[-1]

NOTEBOOK_CODE_HASH = semantic_notebook_code_hash(
    notebook_path
)

prior_code_hash = prior_pipeline_state.get(
    "notebookCodeHash"
)
prior_pipeline_version = prior_pipeline_state.get(
    "pipelineVersion"
)

if (
    ENFORCE_CODE_VERSION_BUMP
    and NOTEBOOK_CODE_HASH
    and prior_code_hash
    and NOTEBOOK_CODE_HASH != prior_code_hash
    and PIPELINE_VERSION == prior_pipeline_version
):
    raise RuntimeError(
        "Notebook code changed but PIPELINE_VERSION was not bumped. "
        "Increment PIPELINE_VERSION and add a CODE_CHANGELOG entry before "
        "running production."
    )

print("Prior successful pipeline version:", prior_pipeline_version)
print("Current pipeline version:", PIPELINE_VERSION)
print(
    "Semantic notebook code hash:",
    NOTEBOOK_CODE_HASH or "UNAVAILABLE"
)

Prior successful pipeline version: 3.1.0
Current pipeline version: 3.1.0
Semantic notebook code hash: 0ec67a2bcf57e45b91145f4e29fa4ac3e0b4d757e6a3789df583ab41b8366069


In [35]:
# 3. Discover the live World Athletics GraphQL endpoint/API key

async def discover_live_wa_api(country_code: str) -> dict[str, Any]:
    env_url = os.getenv("WA_GRAPHQL_URL")
    env_key = os.getenv("WA_API_KEY")

    if env_url and env_key:
        print("Using GraphQL endpoint/API key from environment variables.")
        return {
            "url": env_url,
            "api_key": env_key,
            "headers": {
                "user-agent": os.getenv("WA_USER_AGENT", "Mozilla/5.0"),
                "origin": "https://worldathletics.org",
                "referer": "https://worldathletics.org/",
            }
        }

    captured: dict[str, Any] = {}
    tasks: list[asyncio.Task] = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()
        page = await context.new_page()

        async def inspect_request(request):
            if "graphql" not in request.url.lower():
                return
            try:
                headers = await request.all_headers()
                api_key = headers.get("x-api-key")
                if api_key:
                    captured.update({
                        "url": request.url,
                        "api_key": api_key,
                        "headers": {
                            "x-amz-user-agent": headers.get("x-amz-user-agent"),
                            "user-agent": headers.get("user-agent"),
                            "origin": headers.get("origin") or "https://worldathletics.org",
                            "referer": headers.get("referer") or "https://worldathletics.org/",
                        }
                    })
            except Exception:
                logger.exception("Failed to inspect GraphQL browser request")

        page.on(
            "request",
            lambda request: tasks.append(
                asyncio.create_task(inspect_request(request))
            )
        )

        await page.goto(
            f"https://worldathletics.org/athletes?countryCode={country_code}",
            wait_until="domcontentloaded",
            timeout=90_000
        )
        await page.wait_for_timeout(8_000)

        if tasks:
            await asyncio.gather(*tasks, return_exceptions=True)

        await browser.close()

    if not captured.get("url") or not captured.get("api_key"):
        raise RuntimeError(
            "Could not discover live GraphQL credentials. "
            "Set WA_GRAPHQL_URL and WA_API_KEY as environment variables."
        )

    return captured


wa_api = await discover_live_wa_api(COUNTRY_CODE)
print("GraphQL URL:", wa_api["url"])

GraphQL URL: https://graphql-prod-4895.edge.aws.worldathletics.org/graphql


In [36]:
# 4. Persistent GraphQL client + bounded retry logic

_api_semaphore = asyncio.Semaphore(API_MAX_CONCURRENCY)

_graphql_headers = {"x-api-key": wa_api["api_key"]}
for header_name in ["x-amz-user-agent", "user-agent", "origin", "referer"]:
    value = wa_api.get("headers", {}).get(header_name)
    if value:
        _graphql_headers[header_name] = value

_http_client = httpx.AsyncClient(
    timeout=httpx.Timeout(HTTP_TIMEOUT_SECONDS),
    limits=httpx.Limits(
        max_connections=API_MAX_CONCURRENCY + 4,
        max_keepalive_connections=API_MAX_CONCURRENCY
    ),
    headers=_graphql_headers,
    http2=True,
)

_api_stats = {
    "requests": 0,
    "retries": 0,
    "cache_hits": 0,
}


async def wa_graphql(
    query: str,
    variables: dict[str, Any] | None = None,
    *,
    retries: int = HTTP_MAX_RETRIES
) -> dict[str, Any]:
    variables = variables or {}
    last_error: Exception | None = None

    for attempt in range(retries):
        try:
            async with _api_semaphore:
                _api_stats["requests"] += 1
                response = await _http_client.post(
                    wa_api["url"],
                    json={"query": query, "variables": variables}
                )

            if response.status_code in {429, 500, 502, 503, 504}:
                response.raise_for_status()

            response.raise_for_status()
            payload = response.json()

            if payload.get("errors"):
                raise RuntimeError(
                    json.dumps(payload["errors"], indent=2)
                )

            return payload.get("data") or {}

        except Exception as exc:
            last_error = exc
            if attempt == retries - 1:
                break

            _api_stats["retries"] += 1
            retry_after = None
            if isinstance(exc, httpx.HTTPStatusError):
                retry_after = exc.response.headers.get("Retry-After")

            try:
                delay = float(retry_after) if retry_after else None
            except (TypeError, ValueError):
                delay = None

            if delay is None:
                delay = 2 ** attempt + random.uniform(0.25, 0.9)

            await asyncio.sleep(delay)

    raise last_error or RuntimeError("Unknown GraphQL error")


async def run_bounded(
    items: Iterable[Any],
    worker: Callable[[Any], Awaitable[Any]],
    *,
    concurrency: int,
    label: str,
    progress_every: int = 25,
    progress_seconds: float = 15.0,
):
    items = list(items)
    total = len(items)
    sem = asyncio.Semaphore(concurrency)
    started = time.monotonic()
    last_print = started
    completed = 0
    results = []
    errors = []

    async def wrapped(item):
        async with sem:
            try:
                return item, await worker(item), None
            except Exception as exc:
                return item, None, exc

    tasks = [asyncio.create_task(wrapped(item)) for item in items]

    for task in asyncio.as_completed(tasks):
        item, value, error = await task
        if error is None:
            results.append((item, value))
        else:
            errors.append((item, error))

        completed += 1
        now = time.monotonic()
        if (
            completed == total
            or completed % progress_every == 0
            or now - last_print >= progress_seconds
        ):
            print_progress(label, completed, total, started)
            last_print = now

    return results, errors

In [37]:
# 5. Schema helpers

INTROSPECTION_TYPE_QUERY = """
query IntrospectType($name: String!) {
  __type(name: $name) {
    kind
    name
    fields {
      name
      args {
        name
        type {
          kind name
          ofType { kind name ofType { kind name ofType { kind name } } }
        }
      }
      type {
        kind name
        ofType { kind name ofType { kind name ofType { kind name } } }
      }
    }
  }
}
"""


def graphql_type_string(type_obj):
    if not isinstance(type_obj, dict):
        return None
    kind, name = type_obj.get("kind"), type_obj.get("name")
    if kind == "NON_NULL":
        inner = graphql_type_string(type_obj.get("ofType"))
        return f"{inner}!" if inner else None
    if kind == "LIST":
        inner = graphql_type_string(type_obj.get("ofType"))
        return f"[{inner}]" if inner else None
    return name or kind


def graphql_type_name(type_obj):
    current = type_obj
    while isinstance(current, dict):
        if current.get("name"):
            return current["name"]
        current = current.get("ofType")
    return None


_type_schema_cache: dict[str, dict[str, Any] | None] = {}


async def introspect_type(type_name: str):
    if type_name in _type_schema_cache:
        return _type_schema_cache[type_name]

    data = await wa_graphql(
        INTROSPECTION_TYPE_QUERY,
        {"name": type_name}
    )
    schema = data.get("__type")
    _type_schema_cache[type_name] = schema
    return schema


async def selectable_scalar_fields_for_type(type_name: str):
    schema = await introspect_type(type_name)
    if not schema:
        return []

    output = []
    for field in schema.get("fields") or []:
        base = field.get("type") or {}
        while isinstance(base, dict) and base.get("kind") in {"NON_NULL", "LIST"}:
            base = base.get("ofType") or {}

        if base.get("kind") not in {"SCALAR", "ENUM"}:
            continue

        required_args = [
            arg for arg in (field.get("args") or [])
            if str(graphql_type_string(arg.get("type"))).endswith("!")
        ]
        if not required_args:
            output.append(field["name"])

    return output


query_schema = await introspect_type("Query")
assert query_schema and query_schema.get("fields"), "Could not introspect Query schema."
print("Query fields discovered:", len(query_schema["fields"]))

Query fields discovered: 176


In [38]:
# 6. Stable production queries

RESULTS_QUERY = """
query AthleteResults($id: Int!, $year: Int!) {
  getSingleCompetitorResultsDate(
    id: $id,
    resultsByYear: $year,
    resultsByYearOrderBy: "date"
  ) {
    resultsByDate {
      date competition competitionId venue indoor discipline disciplineCode
      disciplineNameUrlSlug typeNameUrlSlug country category race place mark
      wind notLegal resultScore remark eventId
    }
  }
}
"""

DISCOVERY_PROFILE_QUERY = """
query DiscoveryProfile($id:Int!){
  getSingleCompetitor(id:$id){
    _id
    basicData{
      aaId givenName familyName birthDate countryCode countryFullName male
    }
  }
}
"""

SEARCH_RESULTS_QUERY = """
query SearchResults($query: String!, $start: Int) {
  getSearchResults(query:$query,start:$start) {
    items { title link snippet thumbnail }
  }
}
"""

CALENDAR_DISCOVERY_QUERY = """
query CompetitionCalendar($competitionId: Int, $day: Int) {
  getCalendarCompetitionResults(
    competitionId: $competitionId,
    day: $day
  ) {
    competition {
      dateRange endDate name rankingCategory startDate venue
    }
    eventTitles {
      eventTitle
      rankingCategory
      events {
        gender
        summary {
          mark nationality placeInRace placeInRound points raceNumber records wind
          competitor { name urlSlug id iaafId birthDate hasProfile }
        }
        races {
          date day race raceId raceNumber wind
          results {
            id mark nationality place points qualified records wind remark
            competitor { name urlSlug id iaafId birthDate hasProfile }
          }
        }
      }
    }
    parameters { competitionId day eventId }
  }
}
"""

## 8. Identity and source helpers

`aaAthleteId` remains the canonical athlete-profile key.

Result rows carry two hashes:

- `result_row_hash`: exact source-content identity used for duplicate/change detection.
- `result_identity_key`: a more stable business identity used to classify source corrections as updates where possible.

The current results table remains a clean current snapshot. Historical versions and row-level changes are stored separately under `audit/`, `deltas/` and `versions/`.

In [39]:
# 8. Athlete/profile/result helpers + incremental result merge

RESULT_SOURCE_COLUMNS = [
    "aaAthleteId",
    "date",
    "competition",
    "competitionId",
    "venue",
    "indoor",
    "discipline",
    "disciplineCode",
    "disciplineNameUrlSlug",
    "typeNameUrlSlug",
    "country",
    "category",
    "race",
    "place",
    "mark",
    "wind",
    "notLegal",
    "resultScore",
    "remark",
    "eventId",
]

RESULT_IDENTITY_COLUMNS = [
    "aaAthleteId",
    "date",
    "competitionId",
    "competition",
    "venue",
    "discipline",
    "disciplineCode",
    "category",
    "race",
    "eventId",
]

RESULT_PROVENANCE_COLUMNS = [
    "result_row_hash",
    "result_identity_key",
    "first_seen_run_id",
    "first_seen_at_utc",
    "last_changed_run_id",
    "last_changed_at_utc",
    "pipeline_version",
]


def normalize_mark(value: Any):
    if value is None or pd.isna(value):
        return pd.NA

    return re.sub(
        r"=+$",
        "",
        str(value)
        .strip()
        .upper()
        .replace(" ", "")
    )


def extract_wa_athlete_id(value: Any) -> int | None:
    if value is None or pd.isna(value):
        return None

    value = str(value).strip()

    for pattern in [
        r"/athletes/athlete=(\d+)",
        r"/athletes/_/(\d+)",
        r"-(\d{7,9})(?:[/?#]|$)",
    ]:
        match = re.search(pattern, value)
        if match:
            return int(match.group(1))

    return None


def extract_calendar_aa_id(
    url_slug: Any,
    has_profile: Any
) -> int | None:
    if has_profile is not True:
        return None

    return extract_wa_athlete_id(
        url_slug
    )


def search_seeds_from_profiles(
    df: pd.DataFrame
) -> set[str]:
    seeds = set()

    if "familyName" not in df.columns:
        return seeds

    for family_name in (
        df["familyName"]
        .dropna()
        .astype(str)
    ):
        family_name = family_name.strip()

        if not family_name:
            continue

        seeds.add(
            family_name.upper()
        )

        for token in re.findall(
            r"[A-Za-zÀ-ÖØ-öø-ÿ'-]+",
            family_name
        ):
            token = token.strip().upper()

            if (
                len(token) >= 3
                and token not in EXCLUDE_SEARCH_TOKENS
            ):
                seeds.add(token)

    return seeds


async def fetch_discovery_profile(
    athlete_id: int
):
    data = await wa_graphql(
        DISCOVERY_PROFILE_QUERY,
        {"id": int(athlete_id)}
    )

    athlete = (
        data.get("getSingleCompetitor")
        or {}
    )

    basic = (
        athlete.get("basicData")
        or {}
    )

    male = basic.get("male")

    return {
        "aaAthleteId":
            basic.get("aaId")
            or athlete.get("_id")
            or int(athlete_id),

        "givenName":
            basic.get("givenName"),

        "familyName":
            basic.get("familyName"),

        "birthDate":
            basic.get("birthDate"),

        "countryCode":
            basic.get("countryCode"),

        "countryFullName":
            basic.get("countryFullName"),

        "gender":
            "Male"
            if male is True
            else "Female"
            if male is False
            else None,
    }


async def fetch_athlete_results_api(
    athlete_id: int,
    year: int
):
    data = await wa_graphql(
        RESULTS_QUERY,
        {
            "id": int(athlete_id),
            "year": int(year)
        }
    )

    block = (
        data.get(
            "getSingleCompetitorResultsDate"
        )
        or {}
    )

    return [
        row
        for row in (
            block.get("resultsByDate")
            or []
        )
        if isinstance(row, dict)
    ]


def parse_wa_dates(series: pd.Series) -> pd.Series:
    parsed = pd.to_datetime(
        series,
        format="%d %b %Y",
        errors="coerce",
        utc=True
    )

    missing = parsed.isna()

    if missing.any():
        parsed.loc[missing] = pd.to_datetime(
            series.loc[missing],
            errors="coerce",
            utc=True
        )

    return parsed


def prepare_result_frame(
    df: pd.DataFrame,
    *,
    new_rows: bool = False
) -> pd.DataFrame:

    if df is None or df.empty:
        return pd.DataFrame(
            columns=(
                RESULT_SOURCE_COLUMNS
                + RESULT_PROVENANCE_COLUMNS
            )
        )

    out = df.copy()

    for col in RESULT_SOURCE_COLUMNS:
        if col not in out.columns:
            out[col] = pd.NA

    out["aaAthleteId"] = pd.to_numeric(
        out["aaAthleteId"],
        errors="coerce"
    ).astype("Int64")

    out["result_row_hash"] = out.apply(
        lambda row: stable_record_hash(
            row,
            RESULT_SOURCE_COLUMNS
        ),
        axis=1
    )

    out["result_identity_key"] = out.apply(
        lambda row: stable_record_hash(
            row,
            RESULT_IDENTITY_COLUMNS
        ),
        axis=1
    )

    now_iso = datetime.now(
        timezone.utc
    ).isoformat()

    if "first_seen_run_id" not in out.columns:
        out["first_seen_run_id"] = (
            RUN_ID
            if new_rows
            else "LEGACY_BASELINE"
        )

    if "first_seen_at_utc" not in out.columns:
        out["first_seen_at_utc"] = (
            now_iso
            if new_rows
            else pd.NA
        )

    if "last_changed_run_id" not in out.columns:
        out["last_changed_run_id"] = (
            RUN_ID
            if new_rows
            else "LEGACY_BASELINE"
        )

    if "last_changed_at_utc" not in out.columns:
        out["last_changed_at_utc"] = (
            now_iso
            if new_rows
            else pd.NA
        )

    if "pipeline_version" not in out.columns:
        out["pipeline_version"] = (
            PIPELINE_VERSION
            if new_rows
            else prior_pipeline_version
            or "LEGACY_BASELINE"
        )

    return out


def result_window_cutoff(
    existing_athlete_rows: pd.DataFrame
) -> pd.Timestamp | None:

    if (
        FORCE_FULL_YEAR_RECONCILIATION
        or existing_athlete_rows is None
        or existing_athlete_rows.empty
        or "date" not in existing_athlete_rows.columns
    ):
        return None

    parsed = parse_wa_dates(
        existing_athlete_rows["date"]
    )

    max_date = parsed.max()

    if pd.isna(max_date):
        return None

    return (
        max_date
        - pd.Timedelta(
            days=RESULT_RECONCILIATION_LOOKBACK_DAYS
        )
    )


def merge_one_athlete_results(
    existing_all: pd.DataFrame,
    athlete_id: int,
    incoming_source_rows: list[dict[str, Any]],
    fetched_at_utc: str,
):
    """
    Replace only the athlete's recent reconciliation window.

    Existing rows outside the window are preserved.
    Inside the window, the successful WA response is treated as authoritative.

    Returns:
        merged_current,
        change_events,
        incoming_window_count
    """
    athlete_id = int(athlete_id)

    existing = prepare_result_frame(
        existing_all,
        new_rows=False
    )

    existing_athlete = existing[
        existing["aaAthleteId"]
        .astype("Int64")
        .eq(athlete_id)
    ].copy()

    other_athletes = existing[
        ~existing["aaAthleteId"]
        .astype("Int64")
        .eq(athlete_id)
    ].copy()

    incoming = pd.DataFrame(
        [
            {
                "aaAthleteId":
                    athlete_id,
                **row
            }
            for row in incoming_source_rows
        ]
    )

    incoming = prepare_result_frame(
        incoming,
        new_rows=True
    )

    incoming["source_fetched_at_utc"] = (
        fetched_at_utc
    )

    cutoff = result_window_cutoff(
        existing_athlete
    )

    if cutoff is None:
        existing_window = (
            existing_athlete.copy()
        )
        existing_outside = (
            existing_athlete.iloc[0:0]
            .copy()
        )
        incoming_window = (
            incoming.copy()
        )

    else:
        existing_dates = parse_wa_dates(
            existing_athlete["date"]
        )

        incoming_dates = parse_wa_dates(
            incoming["date"]
        )

        existing_window = existing_athlete[
            existing_dates.ge(cutoff)
        ].copy()

        existing_outside = existing_athlete[
            ~existing_dates.ge(cutoff)
        ].copy()

        # Any incoming row with an unparseable date is retained so it is
        # never silently lost.
        incoming_window = incoming[
            incoming_dates.ge(cutoff)
            | incoming_dates.isna()
        ].copy()

    # Exact duplicates are unchanged and preserve their original provenance.
    existing_by_hash = {
        row["result_row_hash"]: row
        for _, row in existing_window.iterrows()
    }

    incoming_by_hash = {
        row["result_row_hash"]: row
        for _, row in incoming_window.iterrows()
    }

    exact_hashes = (
        set(existing_by_hash)
        & set(incoming_by_hash)
    )

    exact_existing_rows = pd.DataFrame(
        [
            existing_by_hash[h].to_dict()
            for h in exact_hashes
        ]
    )

    removed_existing = existing_window[
        ~existing_window[
            "result_row_hash"
        ].isin(exact_hashes)
    ].copy()

    new_incoming = incoming_window[
        ~incoming_window[
            "result_row_hash"
        ].isin(exact_hashes)
    ].copy()

    # Pair unique identity keys to classify source corrections as UPDATE.
    old_identity_counts = (
        removed_existing[
            "result_identity_key"
        ]
        .value_counts()
        .to_dict()
        if not removed_existing.empty
        else {}
    )

    new_identity_counts = (
        new_incoming[
            "result_identity_key"
        ]
        .value_counts()
        .to_dict()
        if not new_incoming.empty
        else {}
    )

    old_by_identity = {
        row["result_identity_key"]: row
        for _, row in removed_existing.iterrows()
        if old_identity_counts.get(
            row["result_identity_key"]
        ) == 1
    }

    new_by_identity = {
        row["result_identity_key"]: row
        for _, row in new_incoming.iterrows()
        if new_identity_counts.get(
            row["result_identity_key"]
        ) == 1
    }

    update_keys = (
        set(old_by_identity)
        & set(new_by_identity)
    )

    now_iso = datetime.now(
        timezone.utc
    ).isoformat()

    change_events = []
    update_new_hashes = set()
    update_old_hashes = set()

    for identity_key in sorted(
        update_keys
    ):
        old_row = old_by_identity[
            identity_key
        ]
        new_row = new_by_identity[
            identity_key
        ].copy()

        # Preserve first-seen provenance.
        new_row[
            "first_seen_run_id"
        ] = old_row.get(
            "first_seen_run_id"
        )

        new_row[
            "first_seen_at_utc"
        ] = old_row.get(
            "first_seen_at_utc"
        )

        new_row[
            "last_changed_run_id"
        ] = RUN_ID

        new_row[
            "last_changed_at_utc"
        ] = now_iso

        new_row[
            "pipeline_version"
        ] = PIPELINE_VERSION

        new_incoming.loc[
            new_incoming[
                "result_row_hash"
            ].eq(
                new_row[
                    "result_row_hash"
                ]
            ),
            RESULT_PROVENANCE_COLUMNS
        ] = [
            new_row.get(c)
            for c in RESULT_PROVENANCE_COLUMNS
        ]

        update_old_hashes.add(
            old_row["result_row_hash"]
        )

        update_new_hashes.add(
            new_row["result_row_hash"]
        )

        change_events.append({
            "run_id":
                RUN_ID,

            "pipeline_version":
                PIPELINE_VERSION,

            "change_type":
                "UPDATE",

            "aaAthleteId":
                athlete_id,

            "date":
                new_row.get("date"),

            "competitionId":
                new_row.get("competitionId"),

            "discipline":
                new_row.get("discipline"),

            "result_identity_key":
                identity_key,

            "old_result_row_hash":
                old_row.get(
                    "result_row_hash"
                ),

            "new_result_row_hash":
                new_row.get(
                    "result_row_hash"
                ),

            "source_fetched_at_utc":
                fetched_at_utc,

            "changed_at_utc":
                now_iso,
        })

    # New rows not used as updates are inserts.
    insert_rows = new_incoming[
        ~new_incoming[
            "result_row_hash"
        ].isin(update_new_hashes)
    ].copy()

    for _, row in insert_rows.iterrows():
        change_events.append({
            "run_id":
                RUN_ID,

            "pipeline_version":
                PIPELINE_VERSION,

            "change_type":
                "INSERT",

            "aaAthleteId":
                athlete_id,

            "date":
                row.get("date"),

            "competitionId":
                row.get("competitionId"),

            "discipline":
                row.get("discipline"),

            "result_identity_key":
                row.get(
                    "result_identity_key"
                ),

            "old_result_row_hash":
                None,

            "new_result_row_hash":
                row.get(
                    "result_row_hash"
                ),

            "source_fetched_at_utc":
                fetched_at_utc,

            "changed_at_utc":
                now_iso,
        })

    # Removed rows not paired to a replacement are source deletions.
    delete_rows = removed_existing[
        ~removed_existing[
            "result_row_hash"
        ].isin(update_old_hashes)
    ].copy()

    for _, row in delete_rows.iterrows():
        change_events.append({
            "run_id":
                RUN_ID,

            "pipeline_version":
                PIPELINE_VERSION,

            "change_type":
                "DELETE",

            "aaAthleteId":
                athlete_id,

            "date":
                row.get("date"),

            "competitionId":
                row.get("competitionId"),

            "discipline":
                row.get("discipline"),

            "result_identity_key":
                row.get(
                    "result_identity_key"
                ),

            "old_result_row_hash":
                row.get(
                    "result_row_hash"
                ),

            "new_result_row_hash":
                None,

            "source_fetched_at_utc":
                fetched_at_utc,

            "changed_at_utc":
                now_iso,
        })

    # Current authoritative window:
    # unchanged old rows + new source rows (updated or inserted).
    current_window = pd.concat(
        [
            exact_existing_rows,
            new_incoming
        ],
        ignore_index=True,
        sort=False
    )

    merged = pd.concat(
        [
            other_athletes,
            existing_outside,
            current_window
        ],
        ignore_index=True,
        sort=False
    )

    if not merged.empty:
        merged = (
            merged
            .drop_duplicates(
                subset=["result_row_hash"],
                keep="last"
            )
            .reset_index(drop=True)
        )

    return (
        merged,
        pd.DataFrame(change_events),
        len(incoming_window),
    )

In [40]:

# Standard athlete-name enrichment for all athlete-keyed extracts

ATHLETE_NAME_COLUMN = "athleteName"


def build_athlete_name_lookup(
    athlete_master: pd.DataFrame
) -> dict[int, str]:
    """
    Canonical display name keyed by aaAthleteId.
    Preference: athleteName/fullName, then givenName + familyName.
    """
    if athlete_master is None or athlete_master.empty:
        return {}

    master = athlete_master.copy()
    master["aaAthleteId"] = pd.to_numeric(
        master["aaAthleteId"],
        errors="coerce"
    ).astype("Int64")

    def choose_name(row):
        for col in ["athleteName", "fullName"]:
            if col in row.index:
                value = row.get(col)
                if value is not None and not pd.isna(value):
                    value = str(value).strip()
                    if value:
                        return value

        given = str(row.get("givenName") or "").strip()
        family = str(row.get("familyName") or "").strip()

        if given and family:
            return f"{given} {family}"
        return family or given or ""

    master[ATHLETE_NAME_COLUMN] = master.apply(
        choose_name,
        axis=1
    )

    return {
        int(row["aaAthleteId"]): row[ATHLETE_NAME_COLUMN]
        for _, row in master.iterrows()
        if pd.notna(row["aaAthleteId"])
        and str(row[ATHLETE_NAME_COLUMN]).strip()
    }


def attach_athlete_name(
    df: pd.DataFrame,
    athlete_master: pd.DataFrame,
    *,
    id_col: str = "aaAthleteId",
    name_col: str = ATHLETE_NAME_COLUMN,
    fallback_cols=("competitor_name", "name"),
) -> pd.DataFrame:
    if df is None:
        return pd.DataFrame()

    out = df.copy()

    if out.empty or id_col not in out.columns:
        return out

    lookup = build_athlete_name_lookup(
        athlete_master
    )

    ids = pd.to_numeric(
        out[id_col],
        errors="coerce"
    )

    mapped = ids.map(lookup).astype("string")

    if name_col in out.columns:
        existing = (
            out[name_col]
            .astype("string")
            .str.strip()
        )
        mapped = mapped.fillna(
            existing.where(existing.ne(""))
        )

    for col in fallback_cols:
        if col in out.columns:
            fallback = (
                out[col]
                .astype("string")
                .str.strip()
            )
            mapped = mapped.fillna(
                fallback.where(fallback.ne(""))
            )

    out[name_col] = mapped

    cols = list(out.columns)
    cols.remove(name_col)
    cols.insert(
        cols.index(id_col) + 1,
        name_col
    )

    return out[cols]


def ensure_master_athlete_name(
    athlete_master: pd.DataFrame
) -> pd.DataFrame:
    if athlete_master is None or athlete_master.empty:
        return athlete_master.copy()

    out = athlete_master.copy()
    lookup = build_athlete_name_lookup(out)

    ids = pd.to_numeric(
        out["aaAthleteId"],
        errors="coerce"
    )

    out[ATHLETE_NAME_COLUMN] = ids.map(lookup)

    cols = list(out.columns)
    cols.remove(ATHLETE_NAME_COLUMN)
    cols.insert(
        cols.index("aaAthleteId") + 1,
        ATHLETE_NAME_COLUMN
    )

    return out[cols]


## 9. Load current production state

The incremental run requires an existing `athlete_master.parquet`. Other tables can be absent on the first migration run; the notebook will create them.

The hashes recorded here are the **before** state used by table-version audit later.

In [41]:
# 9. Load existing current production tables and baseline hashes

df_all_sgp_athletes = load_current_table(
    "athlete_master"
)

if (
    USE_EXISTING_MASTER_AS_SEED
    and (
        df_all_sgp_athletes is None
        or df_all_sgp_athletes.empty
    )
):
    raise RuntimeError(
        "No athlete_master.parquet production seed exists. "
        "Copy the validated athlete master into "
        f"{OUTPUT_DIR.resolve()} before running incremental production."
    )


df_all_sgp_athletes = ensure_master_athlete_name(
    df_all_sgp_athletes
)

CURRENT_TABLE_NAMES = [
    "athlete_master",
    f"results_{RESULTS_YEAR}",
    "profiles_enriched",
    "personal_bests",
    f"season_bests_{RESULTS_YEAR}",
    "progression",
    "rankings_current",
    "rankings_best",
    "honours",
    "competition_discovery",
    "competition_candidate_audit",
    "competition_exceptions",
    "result_status",
    "result_errors",
    "enrichment_status",
    "enrichment_errors",
    "search_index_audit",
    "search_index_errors",
    "sb_reconciliation",
    "athlete_refresh_state",
    "enrichment_refresh_state",
    "competition_refresh_state",
]

baseline_tables = {
    name: load_current_table(name)
    for name in CURRENT_TABLE_NAMES
}

baseline_table_hashes = {
    name: dataframe_sha256(df)
    for name, df in baseline_tables.items()
}

baseline_row_counts = {
    name: len(df)
    for name, df in baseline_tables.items()
}

df_existing_results = baseline_tables[
    f"results_{RESULTS_YEAR}"
].copy()

df_existing_results = prepare_result_frame(
    df_existing_results,
    new_rows=False
)

df_result_refresh_state = baseline_tables[
    "athlete_refresh_state"
].copy()

df_existing_profiles_enriched = baseline_tables[
    "profiles_enriched"
].copy()

df_enrichment_refresh_state = baseline_tables[
    "enrichment_refresh_state"
].copy()

df_existing_competition_discovery = baseline_tables[
    "competition_discovery"
].copy()

df_existing_competition_exceptions = baseline_tables[
    "competition_exceptions"
].copy()

df_competition_refresh_state = baseline_tables[
    "competition_refresh_state"
].copy()

print(
    "Loaded athlete master:",
    len(df_all_sgp_athletes)
)

print(
    f"Existing results_{RESULTS_YEAR}:",
    len(df_existing_results)
)

print(
    "Existing enriched profiles:",
    len(df_existing_profiles_enriched)
)

Loaded athlete master: 398
Existing results_2026: 1214
Existing enriched profiles: 219


## 10. Optional incremental search-index discovery refresh

This remains **off by default** because the validated master has already undergone full search-index closure.

When enabled, only previously unsearched surname/token seeds and previously unchecked candidate IDs are queried.

In [42]:
SEARCH_AUDIT_CHECKPOINT = CHECKPOINT_DIR / "search_index_audit.parquet"
SEARCH_CANDIDATE_CACHE = CHECKPOINT_DIR / "search_checked_candidate_ids.json"


async def search_results_page(query: str, start: int = 0):
    data = await wa_graphql(
        SEARCH_RESULTS_QUERY,
        {"query": query, "start": start}
    )
    return (data.get("getSearchResults") or {}).get("items") or []


async def crawl_search_index_query(query: str):
    all_items = []
    seen_links = set()
    stale_pages = 0

    for page_num in range(SEARCH_INDEX_MAX_PAGES):
        items = await search_results_page(query, page_num * 10)
        if not items:
            break

        new_count = 0
        for item in items:
            if not isinstance(item, dict):
                continue
            link = str(item.get("link") or "").strip()
            if link and link not in seen_links:
                seen_links.add(link)
                all_items.append(item)
                new_count += 1

        stale_pages = stale_pages + 1 if new_count == 0 else 0
        if stale_pages >= SEARCH_INDEX_STALE_PAGE_LIMIT:
            break

    return all_items


if not RUN_SEARCH_INDEX_REFRESH:
    print(
        "Search-index refresh SKIPPED. "
        f"Using validated master with {len(df_all_sgp_athletes):,} profiles."
    )
    df_search_audit = (
        pd.read_parquet(SEARCH_AUDIT_CHECKPOINT)
        if SEARCH_AUDIT_CHECKPOINT.exists()
        else pd.DataFrame()
    )
    df_search_errors = pd.DataFrame()

else:
    prior_audit = (
        pd.read_parquet(SEARCH_AUDIT_CHECKPOINT)
        if SEARCH_AUDIT_CHECKPOINT.exists()
        else pd.DataFrame()
    )

    previously_searched = set()
    if SEARCH_INCREMENTAL_ONLY and not prior_audit.empty and "query" in prior_audit:
        previously_searched = set(
            prior_audit["query"]
            .dropna()
            .astype(str)
            .str.strip()
            .str.upper()
        )

    checked_candidate_ids = safe_int_set(
        read_json(SEARCH_CANDIDATE_CACHE)
        if SEARCH_CANDIDATE_CACHE.exists()
        else []
    )

    known_ids = safe_int_set(df_all_sgp_athletes["aaAthleteId"])
    checked_candidate_ids |= known_ids

    current_seeds = search_seeds_from_profiles(df_all_sgp_athletes)
    pending = deque(sorted(current_seeds - previously_searched))
    queued = set(pending)
    searched_this_run = set()
    search_audit_rows = []
    search_error_rows = []
    started = time.monotonic()
    batches = 0

    print(
        f"Search-index refresh starting: {len(pending):,} seed(s) "
        f"| prior audited={len(previously_searched):,}"
    )

    while pending:
        batch_queries = []
        while pending and len(batch_queries) < SEARCH_QUERY_CONCURRENCY:
            q = pending.popleft().strip().upper()
            queued.discard(q)
            if q and q not in searched_this_run and q not in previously_searched:
                batch_queries.append(q)
                searched_this_run.add(q)

        if not batch_queries:
            continue

        batches += 1

        search_pairs, batch_errors = await run_bounded(
            batch_queries,
            crawl_search_index_query,
            concurrency=SEARCH_QUERY_CONCURRENCY,
            label=f"Search batch {batches}",
            progress_every=1,
            progress_seconds=10,
        )

        query_to_items = {query: items for query, items in search_pairs}

        for query_text, exc in batch_errors:
            search_error_rows.append({
                "stage": "search",
                "query": query_text,
                "error": repr(exc)
            })

        candidate_sources: dict[int, set[str]] = {}

        for query_text in batch_queries:
            items = query_to_items.get(query_text, [])
            candidate_ids = {
                extract_wa_athlete_id(item.get("link"))
                for item in items
                if "/athletes/" in str(item.get("link") or "").lower()
            }
            candidate_ids.discard(None)

            for athlete_id in candidate_ids:
                candidate_sources.setdefault(int(athlete_id), set()).add(query_text)

            search_audit_rows.append({
                "query": query_text,
                "items": len(items),
                "candidateIds": len(candidate_ids),
                "newCandidates": len(set(candidate_ids) - checked_candidate_ids),
                "newConfirmedSGP": 0,
                "runAt": datetime.now(timezone.utc).isoformat(),
            })

        new_candidate_ids = sorted(
            set(candidate_sources) - checked_candidate_ids
        )

        async def validate_candidate(athlete_id: int):
            return await fetch_discovery_profile(athlete_id)

        validated_pairs, validation_errors = await run_bounded(
            new_candidate_ids,
            validate_candidate,
            concurrency=PROFILE_VALIDATION_CONCURRENCY,
            label="Candidate validation",
            progress_every=25,
            progress_seconds=15,
        )

        newly_confirmed = []
        confirmed_source_queries: dict[int, set[str]] = {}

        for athlete_id, profile in validated_pairs:
            checked_candidate_ids.add(int(athlete_id))
            if (
                str(profile.get("countryCode") or "").strip().upper()
                == COUNTRY_CODE
            ):
                newly_confirmed.append(profile)
                confirmed_source_queries[int(athlete_id)] = candidate_sources.get(int(athlete_id), set())

        for athlete_id, exc in validation_errors:
            search_error_rows.append({
                "stage": "profile_validation",
                "aaAthleteId": int(athlete_id),
                "error": repr(exc)
            })

        if newly_confirmed:
            df_new = pd.DataFrame(newly_confirmed)
            df_all_sgp_athletes = (
                pd.concat(
                    [df_all_sgp_athletes, df_new],
                    ignore_index=True,
                    sort=False
                )
                .drop_duplicates(subset=["aaAthleteId"])
                .reset_index(drop=True)
            )

            new_ids = safe_int_set(df_new["aaAthleteId"])
            known_ids |= new_ids

            for row in search_audit_rows[-len(batch_queries):]:
                q = row["query"]
                row["newConfirmedSGP"] = sum(
                    q in sources
                    for sources in confirmed_source_queries.values()
                )

            for seed in sorted(search_seeds_from_profiles(df_new)):
                if (
                    seed not in searched_this_run
                    and seed not in previously_searched
                    and seed not in queued
                ):
                    pending.append(seed)
                    queued.add(seed)

        # Durable checkpoints every batch.
        combined_audit = pd.concat(
            [prior_audit, pd.DataFrame(search_audit_rows)],
            ignore_index=True,
            sort=False
        ).drop_duplicates(subset=["query"], keep="last")

        save_parquet_atomic(combined_audit, SEARCH_AUDIT_CHECKPOINT)
        write_json_atomic(sorted(checked_candidate_ids), SEARCH_CANDIDATE_CACHE)

        print_progress(
            "Search-index closure",
            len(searched_this_run),
            len(searched_this_run) + len(pending),
            started,
            extra=(
                f"queue={len(pending):,} | profiles={len(df_all_sgp_athletes):,} "
                f"| checked candidates={len(checked_candidate_ids):,}"
            )
        )

    df_search_audit = pd.concat(
        [prior_audit, pd.DataFrame(search_audit_rows)],
        ignore_index=True,
        sort=False
    ).drop_duplicates(subset=["query"], keep="last")

    df_search_errors = pd.DataFrame(search_error_rows)

    print(
        "Search-index refresh complete | "
        f"profiles={len(df_all_sgp_athletes):,} | "
        f"queries this run={len(searched_this_run):,} | "
        f"errors={len(df_search_errors):,}"
    )

Search-index refresh SKIPPED. Using validated master with 398 profiles.


## 11. Select only athlete profiles due for a live result refresh

Selection is persisted in `athlete_refresh_state.parquet`:

- current-year active athletes: due every normal production run by default;
- currently inactive profiles: due every `INACTIVE_RESULT_REFRESH_DAYS`;
- new profiles / profiles never checked: always due.

This prevents repeatedly querying all inactive profiles on every run.

In [43]:
# 11. Select result-refresh population

def refresh_state_map(
    state: pd.DataFrame
) -> dict[int, dict[str, Any]]:
    if (
        state is None
        or state.empty
        or "aaAthleteId" not in state.columns
    ):
        return {}

    output = {}

    for _, row in state.iterrows():
        try:
            athlete_id = int(
                row["aaAthleteId"]
            )
        except Exception:
            continue

        output[athlete_id] = (
            row.to_dict()
        )

    return output


result_state_by_id = refresh_state_map(
    df_result_refresh_state
)

existing_active_ids = safe_int_set(
    df_existing_results.get(
        "aaAthleteId",
        pd.Series(dtype=object)
    )
)

all_profile_ids = sorted(
    safe_int_set(
        df_all_sgp_athletes[
            "aaAthleteId"
        ]
    )
)

now_utc = datetime.now(
    timezone.utc
)

due_profile_ids = []
skipped_profile_ids = []

for athlete_id in all_profile_ids:
    state = result_state_by_id.get(
        athlete_id
    )

    # New/never-checked profile.
    if not state:
        due_profile_ids.append(
            athlete_id
        )
        continue

    last_checked = pd.to_datetime(
        state.get(
            "last_checked_at_utc"
        ),
        errors="coerce",
        utc=True
    )

    if pd.isna(last_checked):
        due_profile_ids.append(
            athlete_id
        )
        continue

    is_active = (
        athlete_id
        in existing_active_ids
    )

    age = (
        now_utc
        - last_checked.to_pydatetime()
    )

    if is_active:
        due = (
            ACTIVE_RESULT_REFRESH_HOURS <= 0
            or age
            >= timedelta(
                hours=ACTIVE_RESULT_REFRESH_HOURS
            )
        )
    else:
        due = (
            age
            >= timedelta(
                days=INACTIVE_RESULT_REFRESH_DAYS
            )
        )

    if due:
        due_profile_ids.append(
            athlete_id
        )
    else:
        skipped_profile_ids.append(
            athlete_id
        )


print(
    "Total profiles:",
    len(all_profile_ids)
)

print(
    "Due for live result refresh:",
    len(due_profile_ids)
)

print(
    "Skipped because recently checked:",
    len(skipped_profile_ids)
)

Total profiles: 398
Due for live result refresh: 219
Skipped because recently checked: 179


## 12. Fetch due athlete histories and merge only the latest delta

Raw API responses are immutable and versioned under:

`raw/results/<year>/<athleteId>/<RUN_ID>.json`

The current result file is **not rebuilt from scratch**. Each successful athlete response only replaces that athlete's recent reconciliation window; unchanged rows retain their original provenance.

In [44]:
# 12. Fetch due athlete histories and apply incremental result deltas

def raw_result_run_path(
    athlete_id: int
) -> Path:
    return (
        RAW_RESULTS_DIR
        / str(int(athlete_id))
        / f"{RUN_ID}.json"
    )


async def result_refresh_worker(
    athlete_id: int
):
    fetched_at = datetime.now(
        timezone.utc
    ).isoformat()

    rows = await fetch_athlete_results_api(
        athlete_id,
        RESULTS_YEAR
    )

    payload = {
        "runId":
            RUN_ID,

        "pipelineVersion":
            PIPELINE_VERSION,

        "aaAthleteId":
            int(athlete_id),

        "year":
            int(RESULTS_YEAR),

        "fetchedAt":
            fetched_at,

        "rows":
            rows,
    }

    write_json_atomic(
        payload,
        raw_result_run_path(
            athlete_id
        )
    )

    return payload


result_pairs, result_task_errors = await run_bounded(
    due_profile_ids,
    result_refresh_worker,
    concurrency=RESULT_FETCH_CONCURRENCY,
    label=f"{RESULTS_YEAR} incremental athlete refresh",
    progress_every=25,
    progress_seconds=15,
) if due_profile_ids else ([], [])


df_results_2026 = (
    df_existing_results.copy()
)

result_change_frames = []
result_refresh_rows = []
result_error_rows = []
changed_result_athlete_ids = set()

source_rows_returned = 0
window_rows_compared = 0

for athlete_id, payload in result_pairs:
    rows = (
        payload.get("rows")
        or []
    )

    fetched_at = payload.get(
        "fetchedAt"
    )

    source_rows_returned += len(
        rows
    )

    (
        df_results_2026,
        athlete_changes,
        athlete_window_count,
    ) = merge_one_athlete_results(
        df_results_2026,
        int(athlete_id),
        rows,
        fetched_at,
    )

    window_rows_compared += (
        athlete_window_count
    )

    if not athlete_changes.empty:
        result_change_frames.append(
            athlete_changes
        )
        changed_result_athlete_ids.add(
            int(athlete_id)
        )

    result_dates = pd.to_datetime(
        pd.Series(
            [
                row.get("date")
                for row in rows
            ]
        ),
        format="%d %b %Y",
        errors="coerce",
        utc=True
    )

    last_result_date = (
        result_dates.max()
        if not result_dates.empty
        else pd.NaT
    )

    result_refresh_rows.append({
        "aaAthleteId":
            int(athlete_id),

        "last_checked_at_utc":
            fetched_at,

        "last_source_row_count":
            len(rows),

        "last_seen_result_date":
            (
                last_result_date.isoformat()
                if pd.notna(
                    last_result_date
                )
                else None
            ),

        "active_current_year":
            bool(rows),

        "last_run_id":
            RUN_ID,

        "pipeline_version":
            PIPELINE_VERSION,
    })


for athlete_id, exc in result_task_errors:
    result_error_rows.append({
        "run_id":
            RUN_ID,

        "aaAthleteId":
            int(athlete_id),

        "stage":
            "result_refresh",

        "error":
            repr(exc),

        "error_at_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    })


df_result_changes = (
    pd.concat(
        result_change_frames,
        ignore_index=True,
        sort=False
    )
    if result_change_frames
    else pd.DataFrame()
)

# Update persistent athlete refresh state only for successful fetches.
if result_refresh_rows:
    df_refresh_new = pd.DataFrame(
        result_refresh_rows
    )

    if not df_result_refresh_state.empty:
        refreshed_ids = safe_int_set(
            df_refresh_new[
                "aaAthleteId"
            ]
        )

        df_result_refresh_state = (
            df_result_refresh_state[
                ~pd.to_numeric(
                    df_result_refresh_state[
                        "aaAthleteId"
                    ],
                    errors="coerce"
                )
                .isin(
                    refreshed_ids
                )
            ]
            .copy()
        )

    df_result_refresh_state = pd.concat(
        [
            df_result_refresh_state,
            df_refresh_new
        ],
        ignore_index=True,
        sort=False
    )


df_result_errors = pd.DataFrame(
    result_error_rows
)

active_2026_ids = sorted(
    safe_int_set(
        df_results_2026.get(
            "aaAthleteId",
            pd.Series(dtype=object)
        )
    )
)

result_counts = (
    df_results_2026[
        "aaAthleteId"
    ]
    .value_counts()
    if not df_results_2026.empty
    else pd.Series(dtype="int64")
)

df_result_status = pd.DataFrame({
    "aaAthleteId":
        all_profile_ids
})

df_result_status[
    "resultRows"
] = (
    df_result_status[
        "aaAthleteId"
    ]
    .map(
        result_counts
    )
    .fillna(0)
    .astype(int)
)

df_result_status[
    "active"
] = (
    df_result_status[
        "resultRows"
    ] > 0
)

df_result_status[
    "run_id"
] = RUN_ID

df_result_status[
    "pipeline_version"
] = PIPELINE_VERSION


print()
print(
    "Source rows returned by WA:",
    source_rows_returned
)

print(
    "Rows inside reconciliation windows:",
    window_rows_compared
)

print(
    "Result change events:",
    len(df_result_changes)
)

if not df_result_changes.empty:
    print(
        df_result_changes[
            "change_type"
        ]
        .value_counts()
        .to_dict()
    )

print(
    "Athletes with result changes:",
    len(changed_result_athlete_ids)
)

print(
    "Current active athletes:",
    len(active_2026_ids)
)

print(
    "Result refresh errors:",
    len(df_result_errors)
)

2026 incremental athlete refresh: 25/219 (11.4%) | elapsed=1s | rate=15.13/s | ETA=12s
2026 incremental athlete refresh: 50/219 (22.8%) | elapsed=2s | rate=17.90/s | ETA=9s
2026 incremental athlete refresh: 75/219 (34.2%) | elapsed=3s | rate=19.74/s | ETA=7s
2026 incremental athlete refresh: 100/219 (45.7%) | elapsed=4s | rate=20.61/s | ETA=5s
2026 incremental athlete refresh: 125/219 (57.1%) | elapsed=5s | rate=20.84/s | ETA=4s
2026 incremental athlete refresh: 150/219 (68.5%) | elapsed=7s | rate=20.86/s | ETA=3s
2026 incremental athlete refresh: 175/219 (79.9%) | elapsed=8s | rate=20.78/s | ETA=2s
2026 incremental athlete refresh: 200/219 (91.3%) | elapsed=9s | rate=21.30/s | ETA=0s
2026 incremental athlete refresh: 219/219 (100.0%) | elapsed=10s | rate=21.46/s | ETA=0s

Source rows returned by WA: 1218
Rows inside reconciliation windows: 467
Result change events: 4
{'INSERT': 4}
Athletes with result changes: 4
Current active athletes: 219
Result refresh errors: 0


## 13. Process only newly encountered competitions

Previously processed competition IDs are loaded from `competition_refresh_state`, existing discovery output and existing exceptions.

Only competition IDs newly introduced by the current result snapshot are queued. If a new SGP athlete is discovered, that athlete's current-year history is fetched immediately and any additional new competition IDs are added recursively.

In [45]:
# 13A. Competition helpers

def raw_competition_run_path(
    competition_id: int
) -> Path:
    return (
        RAW_COMPETITION_DIR
        / str(int(competition_id))
        / f"{RUN_ID}.json"
    )


async def fetch_competition_calendar_live(
    competition_id: int
):
    fetched_at = datetime.now(
        timezone.utc
    ).isoformat()

    data = await wa_graphql(
        CALENDAR_DISCOVERY_QUERY,
        {
            "competitionId":
                int(competition_id),

            "day":
                None
        }
    )

    block = (
        data.get(
            "getCalendarCompetitionResults"
        )
        or {}
    )

    write_json_atomic(
        {
            "runId":
                RUN_ID,

            "pipelineVersion":
                PIPELINE_VERSION,

            "competitionId":
                int(competition_id),

            "fetchedAt":
                fetched_at,

            "block":
                block,
        },
        raw_competition_run_path(
            competition_id
        )
    )

    return {
        "block":
            block,

        "fetchedAt":
            fetched_at,
    }


def flatten_calendar_sgp(
    calendar_block,
    competition_id
):
    rows = []

    comp = (
        calendar_block.get(
            "competition"
        )
        or {}
    )

    def append_result(
        result,
        group,
        event,
        source,
        race=None
    ):
        nationality = str(
            result.get(
                "nationality"
            )
            or ""
        ).strip().upper()

        if nationality not in {
            "SGP",
            "SIN",
            "SINGAPORE"
        }:
            return

        competitor = (
            result.get(
                "competitor"
            )
            or {}
        )

        has_profile = (
            competitor.get(
                "hasProfile"
            )
        )

        rows.append({
            "competitionId":
                int(
                    competition_id
                ),

            "competition":
                comp.get("name"),

            "source":
                source,

            "eventTitle":
                group.get(
                    "eventTitle"
                ),

            "rankingCategory":
                group.get(
                    "rankingCategory"
                ),

            "gender":
                event.get("gender"),

            "race":
                race.get("race")
                if race
                else None,

            "raceId":
                race.get("raceId")
                if race
                else None,

            "competitor_internal_id":
                competitor.get("id"),

            "competitor_iaafId":
                competitor.get(
                    "iaafId"
                ),

            "competitor_name":
                competitor.get("name"),

            "competitor_urlSlug":
                competitor.get(
                    "urlSlug"
                ),

            "birthDate":
                competitor.get(
                    "birthDate"
                ),

            "hasProfile":
                has_profile,

            "candidate_aaAthleteId":
                extract_calendar_aa_id(
                    competitor.get(
                        "urlSlug"
                    ),
                    has_profile
                ),

            "nationality":
                nationality,

            "mark":
                result.get("mark"),

            "first_seen_run_id":
                RUN_ID,

            "first_seen_at_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),

            "pipeline_version":
                PIPELINE_VERSION,
        })

    for group in (
        calendar_block.get(
            "eventTitles"
        )
        or []
    ):
        for event in (
            group.get("events")
            or []
        ):
            for result in (
                event.get("summary")
                or []
            ):
                append_result(
                    result,
                    group,
                    event,
                    "summary"
                )

            for race in (
                event.get("races")
                or []
            ):
                for result in (
                    race.get("results")
                    or []
                ):
                    append_result(
                        result,
                        group,
                        event,
                        "race",
                        race
                    )

    return rows

In [46]:
# 13B. New-competition-only recursive closure

df_competition_sgp = (
    df_existing_competition_discovery
    .copy()
)

df_competition_exceptions = (
    df_existing_competition_exceptions
    .copy()
)

df_competition_candidate_audit = (
    baseline_tables[
        "competition_candidate_audit"
    ].copy()
)

competition_refresh_rows = []
new_competition_rows = []
new_competition_exception_rows = []
new_candidate_audit_rows = []

known_profile_ids = safe_int_set(
    df_all_sgp_athletes[
        "aaAthleteId"
    ]
)

new_profile_ids = set()

# Processed IDs from durable state.
processed_competition_ids = set()

if (
    not df_competition_refresh_state.empty
    and "competitionId"
    in df_competition_refresh_state.columns
):
    processed_competition_ids |= safe_int_set(
        df_competition_refresh_state[
            "competitionId"
        ]
    )

# Backward-compatible derivation from legacy outputs.
if (
    not df_existing_competition_discovery.empty
    and "competitionId"
    in df_existing_competition_discovery.columns
):
    processed_competition_ids |= safe_int_set(
        df_existing_competition_discovery[
            "competitionId"
        ]
    )

if (
    not df_existing_competition_exceptions.empty
    and "competitionId"
    in df_existing_competition_exceptions.columns
):
    processed_competition_ids |= safe_int_set(
        df_existing_competition_exceptions[
            "competitionId"
        ]
    )


current_result_competition_ids = safe_int_set(
    df_results_2026.get(
        "competitionId",
        pd.Series(dtype=object)
    )
)

pending = deque(
    sorted(
        current_result_competition_ids
        - processed_competition_ids
    )
)

queued = set(
    pending
)

competition_started = (
    time.monotonic()
)

print(
    "Previously accounted competitions:",
    len(processed_competition_ids)
)

print(
    "New competition IDs queued:",
    len(pending)
)


async def fetch_new_athlete_results_and_merge(
    athlete_id: int
):
    global df_results_2026

    fetched_at = datetime.now(
        timezone.utc
    ).isoformat()

    rows = await fetch_athlete_results_api(
        athlete_id,
        RESULTS_YEAR
    )

    write_json_atomic(
        {
            "runId":
                RUN_ID,

            "pipelineVersion":
                PIPELINE_VERSION,

            "aaAthleteId":
                int(athlete_id),

            "year":
                int(RESULTS_YEAR),

            "fetchedAt":
                fetched_at,

            "rows":
                rows,
        },
        raw_result_run_path(
            athlete_id
        )
    )

    (
        df_results_2026,
        changes,
        _
    ) = merge_one_athlete_results(
        df_results_2026,
        int(athlete_id),
        rows,
        fetched_at,
    )

    return {
        "rows":
            rows,

        "changes":
            changes,

        "fetchedAt":
            fetched_at,
    }


while pending:
    batch_ids = []

    while (
        pending
        and len(batch_ids)
        < COMPETITION_FETCH_CONCURRENCY
    ):
        competition_id = int(
            pending.popleft()
        )

        queued.discard(
            competition_id
        )

        if (
            competition_id
            in processed_competition_ids
        ):
            continue

        if (
            competition_id
            in KNOWN_COMPETITION_EXCEPTIONS
            and not RETRY_KNOWN_COMPETITION_EXCEPTIONS
        ):
            now_iso = datetime.now(
                timezone.utc
            ).isoformat()

            exception_row = {
                "competitionId":
                    competition_id,

                "status":
                    "WA_BACKEND_UNRESOLVED",

                "knownSourceException":
                    True,

                "error":
                    (
                        "Known World Athletics calendar backend "
                        "exception; retry skipped."
                    ),

                "closureComplete":
                    False,

                "run_id":
                    RUN_ID,

                "pipeline_version":
                    PIPELINE_VERSION,

                "recorded_at_utc":
                    now_iso,
            }

            new_competition_exception_rows.append(
                exception_row
            )

            competition_refresh_rows.append({
                "competitionId":
                    competition_id,

                "status":
                    "KNOWN_EXCEPTION",

                "last_checked_at_utc":
                    now_iso,

                "last_run_id":
                    RUN_ID,

                "pipeline_version":
                    PIPELINE_VERSION,
            })

            processed_competition_ids.add(
                competition_id
            )

            continue

        batch_ids.append(
            competition_id
        )

    if not batch_ids:
        continue

    comp_pairs, comp_errors = await run_bounded(
        batch_ids,
        fetch_competition_calendar_live,
        concurrency=COMPETITION_FETCH_CONCURRENCY,
        label="New competition discovery",
        progress_every=1,
        progress_seconds=10,
    )

    candidate_sources: dict[
        int,
        set[int]
    ] = {}

    for competition_id, payload in comp_pairs:
        competition_id = int(
            competition_id
        )

        block = payload["block"]

        rows = flatten_calendar_sgp(
            block,
            competition_id
        )

        new_competition_rows.extend(
            rows
        )

        now_iso = datetime.now(
            timezone.utc
        ).isoformat()

        competition_refresh_rows.append({
            "competitionId":
                competition_id,

            "status":
                "SUCCESS",

            "last_checked_at_utc":
                now_iso,

            "last_run_id":
                RUN_ID,

            "pipeline_version":
                PIPELINE_VERSION,

            "sgp_rows":
                len(rows),
        })

        processed_competition_ids.add(
            competition_id
        )

        for row in rows:
            athlete_id = row.get(
                "candidate_aaAthleteId"
            )

            if (
                row.get("hasProfile")
                is True
                and athlete_id
                is not None
            ):
                candidate_sources.setdefault(
                    int(athlete_id),
                    set()
                ).add(
                    competition_id
                )

    for competition_id, exc in comp_errors:
        competition_id = int(
            competition_id
        )

        now_iso = datetime.now(
            timezone.utc
        ).isoformat()

        exception_row = {
            "competitionId":
                competition_id,

            "status":
                "WA_BACKEND_UNRESOLVED",

            "knownSourceException":
                (
                    competition_id
                    in KNOWN_COMPETITION_EXCEPTIONS
                ),

            "error":
                repr(exc),

            "closureComplete":
                False,

            "run_id":
                RUN_ID,

            "pipeline_version":
                PIPELINE_VERSION,

            "recorded_at_utc":
                now_iso,
        }

        new_competition_exception_rows.append(
            exception_row
        )

        competition_refresh_rows.append({
            "competitionId":
                competition_id,

            "status":
                "ERROR",

            "last_checked_at_utc":
                now_iso,

            "last_run_id":
                RUN_ID,

            "pipeline_version":
                PIPELINE_VERSION,

            "error":
                repr(exc),
        })

        processed_competition_ids.add(
            competition_id
        )

    new_candidate_ids = sorted(
        set(candidate_sources)
        - known_profile_ids
    )

    async def candidate_profile_worker(
        athlete_id: int
    ):
        return await fetch_discovery_profile(
            athlete_id
        )

    profile_pairs, profile_errors = await run_bounded(
        new_candidate_ids,
        candidate_profile_worker,
        concurrency=PROFILE_VALIDATION_CONCURRENCY,
        label="New competition candidate profiles",
        progress_every=10,
        progress_seconds=15,
    ) if new_candidate_ids else ([], [])

    confirmed_new_profiles = []

    for athlete_id, profile in profile_pairs:
        athlete_id = int(
            athlete_id
        )

        is_sgp = (
            str(
                profile.get(
                    "countryCode"
                )
                or ""
            )
            .strip()
            .upper()
            == COUNTRY_CODE
        )

        new_candidate_audit_rows.append({
            "run_id":
                RUN_ID,

            "sourceCompetitionIds":
                ";".join(
                    map(
                        str,
                        sorted(
                            candidate_sources.get(
                                athlete_id,
                                set()
                            )
                        )
                    )
                ),

            "aaAthleteId":
                athlete_id,

            "confirmedSGP":
                is_sgp,

            "profileValidationError":
                None,

            "pipeline_version":
                PIPELINE_VERSION,
        })

        if is_sgp:
            confirmed_new_profiles.append(
                profile
            )

    for athlete_id, exc in profile_errors:
        new_candidate_audit_rows.append({
            "run_id":
                RUN_ID,

            "aaAthleteId":
                int(athlete_id),

            "sourceCompetitionIds":
                ";".join(
                    map(
                        str,
                        sorted(
                            candidate_sources.get(
                                int(
                                    athlete_id
                                ),
                                set()
                            )
                        )
                    )
                ),

            "confirmedSGP":
                False,

            "profileValidationError":
                repr(exc),

            "pipeline_version":
                PIPELINE_VERSION,
        })

    if confirmed_new_profiles:
        df_new_profiles = pd.DataFrame(
            confirmed_new_profiles
        )

        df_new_profiles[
            "first_seen_run_id"
        ] = RUN_ID

        df_new_profiles[
            "first_seen_at_utc"
        ] = datetime.now(
            timezone.utc
        ).isoformat()

        df_new_profiles[
            "pipeline_version"
        ] = PIPELINE_VERSION

        df_all_sgp_athletes = pd.concat(
            [
                df_all_sgp_athletes,
                df_new_profiles
            ],
            ignore_index=True,
            sort=False
        )

        df_all_sgp_athletes = (
            df_all_sgp_athletes
            .drop_duplicates(
                subset=["aaAthleteId"],
                keep="last"
            )
            .reset_index(drop=True)
        )

        batch_new_profile_ids = safe_int_set(
            df_new_profiles[
                "aaAthleteId"
            ]
        )

        new_profile_ids |= (
            batch_new_profile_ids
        )

        known_profile_ids |= (
            batch_new_profile_ids
        )

        # Fetch each newly discovered athlete's current-year history.
        async def new_athlete_result_worker(
            athlete_id: int
        ):
            return await fetch_new_athlete_results_and_merge(
                athlete_id
            )

        new_result_pairs, new_result_errors = await run_bounded(
            sorted(
                batch_new_profile_ids
            ),
            new_athlete_result_worker,
            concurrency=RESULT_FETCH_CONCURRENCY,
            label="New athlete result histories",
            progress_every=5,
            progress_seconds=10,
        )

        for athlete_id, payload in new_result_pairs:
            athlete_id = int(
                athlete_id
            )

            changes = payload[
                "changes"
            ]

            if not changes.empty:
                result_change_frames.append(
                    changes
                )
                changed_result_athlete_ids.add(
                    athlete_id
                )

            new_results = payload[
                "rows"
            ]

            for new_comp_id in safe_int_set(
                [
                    row.get(
                        "competitionId"
                    )
                    for row in new_results
                ]
            ):
                if (
                    new_comp_id
                    not in processed_competition_ids
                    and new_comp_id
                    not in queued
                ):
                    pending.append(
                        new_comp_id
                    )
                    queued.add(
                        new_comp_id
                    )

        for athlete_id, exc in new_result_errors:
            result_error_rows.append({
                "run_id":
                    RUN_ID,

                "aaAthleteId":
                    int(athlete_id),

                "stage":
                    "new_athlete_result_fetch",

                "error":
                    repr(exc),

                "error_at_utc":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
            })

    print_progress(
        "Competition incremental closure",
        len(
            processed_competition_ids
        ),
        (
            len(
                processed_competition_ids
            )
            + len(pending)
        ),
        competition_started,
        extra=(
            f"queue={len(pending):,} "
            f"| new profiles={len(new_profile_ids):,}"
        )
    )


# Append only newly processed competition rows.
if new_competition_rows:
    df_new_competition_rows = pd.DataFrame(
        new_competition_rows
    )

    df_competition_sgp = pd.concat(
        [
            df_competition_sgp,
            df_new_competition_rows
        ],
        ignore_index=True,
        sort=False
    )

    df_competition_sgp = (
        df_competition_sgp
        .drop_duplicates()
        .reset_index(drop=True)
    )
else:
    df_new_competition_rows = pd.DataFrame()


if new_competition_exception_rows:
    df_new_competition_exceptions = (
        pd.DataFrame(
            new_competition_exception_rows
        )
    )

    df_competition_exceptions = pd.concat(
        [
            df_competition_exceptions,
            df_new_competition_exceptions
        ],
        ignore_index=True,
        sort=False
    )
else:
    df_new_competition_exceptions = (
        pd.DataFrame()
    )


if new_candidate_audit_rows:
    df_new_candidate_audit = pd.DataFrame(
        new_candidate_audit_rows
    )

    df_competition_candidate_audit = pd.concat(
        [
            df_competition_candidate_audit,
            df_new_candidate_audit
        ],
        ignore_index=True,
        sort=False
    )
else:
    df_new_candidate_audit = pd.DataFrame()


if competition_refresh_rows:
    df_comp_refresh_new = pd.DataFrame(
        competition_refresh_rows
    )

    if not df_competition_refresh_state.empty:
        refreshed_comp_ids = safe_int_set(
            df_comp_refresh_new[
                "competitionId"
            ]
        )

        df_competition_refresh_state = (
            df_competition_refresh_state[
                ~pd.to_numeric(
                    df_competition_refresh_state[
                        "competitionId"
                    ],
                    errors="coerce"
                )
                .isin(
                    refreshed_comp_ids
                )
            ]
            .copy()
        )

    df_competition_refresh_state = pd.concat(
        [
            df_competition_refresh_state,
            df_comp_refresh_new
        ],
        ignore_index=True,
        sort=False
    )


# Rebuild final result-change dataframe after recursive discoveries.
df_result_changes = (
    pd.concat(
        result_change_frames,
        ignore_index=True,
        sort=False
    )
    if result_change_frames
    else pd.DataFrame()
)

df_result_errors = pd.DataFrame(
    result_error_rows
)

active_2026_ids = sorted(
    safe_int_set(
        df_results_2026.get(
            "aaAthleteId",
            pd.Series(dtype=object)
        )
    )
)

print()
print(
    "New competitions processed this run:",
    len(
        set(
            df_comp_refresh_new[
                "competitionId"
            ]
        )
        if competition_refresh_rows
        else set()
    )
)

print(
    "New profiles discovered:",
    len(new_profile_ids)
)

print(
    "New competition result rows appended:",
    len(df_new_competition_rows)
)

print(
    "Current profiles:",
    len(df_all_sgp_athletes)
)

print(
    "Current active athletes:",
    len(active_2026_ids)
)

Previously accounted competitions: 134
New competition IDs queued: 0

New competitions processed this run: 0
New profiles discovered: 0
New competition result rows appended: 0
Current profiles: 398
Current active athletes: 219


## 14. Build enrichment query and select only athletes due for enrichment refresh

An athlete is refreshed when:

- their results changed in this run;
- they were newly discovered;
- they are active but missing from `profiles_enriched`; or
- their enrichment is older than `ENRICHMENT_REFRESH_DAYS`.

All other enrichment rows are carried forward unchanged.

In [47]:
async def build_enrichment_query():
    type_names = [
        "AthleteNewData",
        "personalBestsType",
        "seasonsBestsType",
        "singleCompetitorResult",
        "singleCompetitorProgression",
        "singleCompetitorProgressionResult",
        "worldRankingCurrent",
        "worldRankingBest",
        "honour",
    ]

    field_lists = await asyncio.gather(*[
        selectable_scalar_fields_for_type(name)
        for name in type_names
    ])

    (
        basic_fields,
        pb_wrapper,
        sb_wrapper,
        result_fields,
        prog_fields,
        prog_result_fields,
        current_fields,
        best_fields,
        honour_fields,
    ) = field_lists

    def block(fields, spaces=8):
        pad = " " * spaces
        return "\n".join(pad + field for field in fields)

    return f"""
query FullAthlete($id: Int!) {{
  getSingleCompetitor(id: $id) {{
    _id
    basicData {{
{block(basic_fields)}
    }}
    personalBests {{
{block(pb_wrapper)}
      results {{
{block(result_fields)}
      }}
    }}
    seasonsBests {{
{block(sb_wrapper)}
      results {{
{block(result_fields)}
      }}
    }}
    progressionOfSeasonsBests {{
{block(prog_fields)}
      results {{
{block(prog_result_fields)}
      }}
    }}
    worldRankings {{
      current {{
{block(current_fields)}
      }}
      best {{
{block(best_fields)}
      }}
    }}
    honours {{
{block(honour_fields)}
    }}
  }}
}}
"""


FINAL_ENRICHMENT_QUERY = await build_enrichment_query()


async def fetch_full_athlete_enrichment(athlete_id: int):
    data = await wa_graphql(
        FINAL_ENRICHMENT_QUERY,
        {"id": int(athlete_id)}
    )
    return data.get("getSingleCompetitor") or {}

In [48]:
# 14B. Select targeted enrichment refresh population

def enrichment_state_map(
    state: pd.DataFrame
) -> dict[int, dict[str, Any]]:
    if (
        state is None
        or state.empty
        or "aaAthleteId"
        not in state.columns
    ):
        return {}

    output = {}

    for _, row in state.iterrows():
        try:
            athlete_id = int(
                row[
                    "aaAthleteId"
                ]
            )
        except Exception:
            continue

        output[
            athlete_id
        ] = row.to_dict()

    return output


enrichment_state_by_id = enrichment_state_map(
    df_enrichment_refresh_state
)

enriched_profile_ids = safe_int_set(
    df_existing_profiles_enriched.get(
        "aaAthleteId",
        pd.Series(dtype=object)
    )
)

active_id_set = set(
    active_2026_ids
)

enrichment_refresh_ids = set(
    changed_result_athlete_ids
) | set(
    new_profile_ids
) | (
    active_id_set
    - enriched_profile_ids
)

now_utc = datetime.now(
    timezone.utc
)

for athlete_id in active_id_set:
    state = enrichment_state_by_id.get(
        athlete_id
    )

    if not state:
        # If legacy enrichment exists, infer recency from its raw checkpoint
        # where possible; otherwise refresh.
        legacy_raw = (
            RAW_ENRICHMENT_DIR
            / f"{athlete_id}.json"
        )

        if legacy_raw.exists():
            modified_at = datetime.fromtimestamp(
                legacy_raw.stat().st_mtime,
                tz=timezone.utc
            )

            if (
                now_utc
                - modified_at
                >= timedelta(
                    days=ENRICHMENT_REFRESH_DAYS
                )
            ):
                enrichment_refresh_ids.add(
                    athlete_id
                )
        else:
            enrichment_refresh_ids.add(
                athlete_id
            )

        continue

    last_refreshed = pd.to_datetime(
        state.get(
            "last_refreshed_at_utc"
        ),
        errors="coerce",
        utc=True
    )

    if (
        pd.isna(last_refreshed)
        or (
            now_utc
            - last_refreshed.to_pydatetime()
            >= timedelta(
                days=ENRICHMENT_REFRESH_DAYS
            )
        )
    ):
        enrichment_refresh_ids.add(
            athlete_id
        )


enrichment_refresh_ids = sorted(
    enrichment_refresh_ids
)

print(
    "Active athletes:",
    len(active_id_set)
)

print(
    "Enrichment refresh required:",
    len(enrichment_refresh_ids)
)

print(
    "Triggered by result/profile changes:",
    len(
        (
            set(
                changed_result_athlete_ids
            )
            | set(
                new_profile_ids
            )
        )
        & set(
            enrichment_refresh_ids
        )
    )
)

Active athletes: 219
Enrichment refresh required: 4
Triggered by result/profile changes: 4


## 15. Fetch only targeted enrichment and replace only those athlete partitions

The enrichment tables are mutable derived views (PBs, SBs, rankings, progression). For an athlete that is refreshed, that athlete's partition is replaced with the latest World Athletics representation. Other athletes' rows remain untouched.

Immutable before/after table versions are captured later.

In [49]:
# 15A. Fetch targeted enrichment

def raw_enrichment_run_path(
    athlete_id: int
) -> Path:
    return (
        RAW_ENRICHMENT_DIR
        / str(int(athlete_id))
        / f"{RUN_ID}.json"
    )


async def enrichment_worker(
    athlete_id: int
):
    fetched_at = datetime.now(
        timezone.utc
    ).isoformat()

    payload = await fetch_full_athlete_enrichment(
        athlete_id
    )

    write_json_atomic(
        {
            "runId":
                RUN_ID,

            "pipelineVersion":
                PIPELINE_VERSION,

            "aaAthleteId":
                int(athlete_id),

            "fetchedAt":
                fetched_at,

            "payload":
                payload,
        },
        raw_enrichment_run_path(
            athlete_id
        )
    )

    return {
        "payload":
            payload,

        "fetchedAt":
            fetched_at,
    }


enrichment_pairs, enrichment_task_errors = await run_bounded(
    enrichment_refresh_ids,
    enrichment_worker,
    concurrency=ENRICHMENT_FETCH_CONCURRENCY,
    label="Incremental enrichment refresh",
    progress_every=20,
    progress_seconds=15,
) if enrichment_refresh_ids else ([], [])


enrichment_payloads = {
    int(
        athlete_id
    ): payload[
        "payload"
    ]
    for athlete_id, payload
    in enrichment_pairs
}

enrichment_fetch_meta = {
    int(
        athlete_id
    ): payload[
        "fetchedAt"
    ]
    for athlete_id, payload
    in enrichment_pairs
}

enrichment_error_rows = [
    {
        "run_id":
            RUN_ID,

        "aaAthleteId":
            int(athlete_id),

        "stage":
            "enrichment_refresh",

        "error":
            repr(exc),

        "error_at_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }
    for athlete_id, exc
    in enrichment_task_errors
]

df_enrichment_errors = pd.DataFrame(
    enrichment_error_rows
)

print(
    "Enrichment payloads fetched:",
    len(enrichment_payloads)
)

print(
    "Enrichment errors:",
    len(df_enrichment_errors)
)

Incremental enrichment refresh: 4/4 (100.0%) | elapsed=0s | rate=6.22/s | ETA=0s
Enrichment payloads fetched: 4
Enrichment errors: 0


In [50]:
# 15B. Flatten only refreshed enrichment payloads

def scalar_dict(obj):
    return {
        k: v
        for k, v in obj.items()
        if isinstance(
            v,
            (
                str,
                int,
                float,
                bool,
                type(None)
            )
        )
    }


profiles = []
pbs = []
sbs = []
progression = []
rank_current = []
rank_best = []
honours = []

for athlete_id, athlete in (
    enrichment_payloads.items()
):
    basic = (
        athlete.get(
            "basicData"
        )
        or {}
    )

    male = basic.get("male")

    profiles.append({
        "aaAthleteId":
            athlete_id,

        "graphQL_id":
            athlete.get("_id"),

        "gender":
            "Male"
            if male is True
            else "Female"
            if male is False
            else None,

        "last_refreshed_run_id":
            RUN_ID,

        "last_refreshed_at_utc":
            enrichment_fetch_meta.get(
                athlete_id
            ),

        "pipeline_version":
            PIPELINE_VERSION,

        **scalar_dict(basic)
    })

    for result in (
        athlete.get(
            "personalBests"
        )
        or {}
    ).get("results") or []:
        if isinstance(
            result,
            dict
        ):
            pbs.append({
                "aaAthleteId":
                    athlete_id,
                **result
            })

    for result in (
        athlete.get(
            "seasonsBests"
        )
        or {}
    ).get("results") or []:
        if isinstance(
            result,
            dict
        ):
            sbs.append({
                "aaAthleteId":
                    athlete_id,
                **result
            })

    for event_block in (
        athlete.get(
            "progressionOfSeasonsBests"
        )
        or []
    ):
        if not isinstance(
            event_block,
            dict
        ):
            continue

        event_meta = {
            k: v
            for k, v
            in event_block.items()
            if k != "results"
        }

        for result in (
            event_block.get(
                "results"
            )
            or []
        ):
            if isinstance(
                result,
                dict
            ):
                progression.append({
                    "aaAthleteId":
                        athlete_id,
                    **event_meta,
                    **result
                })

    rankings = (
        athlete.get(
            "worldRankings"
        )
        or {}
    )

    for row in (
        rankings.get(
            "current"
        )
        or []
    ):
        if isinstance(
            row,
            dict
        ):
            rank_current.append({
                "aaAthleteId":
                    athlete_id,
                **row
            })

    for row in (
        rankings.get(
            "best"
        )
        or []
    ):
        if isinstance(
            row,
            dict
        ):
            rank_best.append({
                "aaAthleteId":
                    athlete_id,
                **row
            })

    for row in (
        athlete.get(
            "honours"
        )
        or []
    ):
        if isinstance(
            row,
            dict
        ):
            honours.append({
                "aaAthleteId":
                    athlete_id,
                **row
            })


fresh_enrichment_tables = {
    "profiles_enriched":
        pd.DataFrame(
            profiles
        ),

    "personal_bests":
        pd.DataFrame(
            pbs
        ),

    f"season_bests_{RESULTS_YEAR}":
        pd.DataFrame(
            sbs
        ),

    "progression":
        pd.DataFrame(
            progression
        ),

    "rankings_current":
        pd.DataFrame(
            rank_current
        ),

    "rankings_best":
        pd.DataFrame(
            rank_best
        ),

    "honours":
        pd.DataFrame(
            honours
        ),
}


def replace_athlete_partition(
    existing: pd.DataFrame,
    fresh: pd.DataFrame,
    refreshed_ids: set[int]
) -> pd.DataFrame:

    existing = (
        existing.copy()
        if existing is not None
        else pd.DataFrame()
    )

    if not refreshed_ids:
        return existing

    if (
        not existing.empty
        and "aaAthleteId"
        in existing.columns
    ):
        existing_ids = pd.to_numeric(
            existing[
                "aaAthleteId"
            ],
            errors="coerce"
        )

        existing = existing[
            ~existing_ids.isin(
                refreshed_ids
            )
        ].copy()

    return pd.concat(
        [
            existing,
            fresh
        ],
        ignore_index=True,
        sort=False
    )


refreshed_id_set = set(
    enrichment_payloads
)

df_profiles_enriched = replace_athlete_partition(
    baseline_tables[
        "profiles_enriched"
    ],
    fresh_enrichment_tables[
        "profiles_enriched"
    ],
    refreshed_id_set,
)

df_personal_bests = replace_athlete_partition(
    baseline_tables[
        "personal_bests"
    ],
    fresh_enrichment_tables[
        "personal_bests"
    ],
    refreshed_id_set,
)

df_season_bests_2026 = replace_athlete_partition(
    baseline_tables[
        f"season_bests_{RESULTS_YEAR}"
    ],
    fresh_enrichment_tables[
        f"season_bests_{RESULTS_YEAR}"
    ],
    refreshed_id_set,
)

df_progression = replace_athlete_partition(
    baseline_tables[
        "progression"
    ],
    fresh_enrichment_tables[
        "progression"
    ],
    refreshed_id_set,
)

df_rankings_current = replace_athlete_partition(
    baseline_tables[
        "rankings_current"
    ],
    fresh_enrichment_tables[
        "rankings_current"
    ],
    refreshed_id_set,
)

df_rankings_best = replace_athlete_partition(
    baseline_tables[
        "rankings_best"
    ],
    fresh_enrichment_tables[
        "rankings_best"
    ],
    refreshed_id_set,
)

df_honours = replace_athlete_partition(
    baseline_tables[
        "honours"
    ],
    fresh_enrichment_tables[
        "honours"
    ],
    refreshed_id_set,
)


# Persist enrichment refresh state.
enrichment_refresh_rows = []

for athlete_id in refreshed_id_set:
    enrichment_refresh_rows.append({
        "aaAthleteId":
            int(
                athlete_id
            ),

        "last_refreshed_at_utc":
            enrichment_fetch_meta.get(
                athlete_id
            ),

        "last_run_id":
            RUN_ID,

        "pipeline_version":
            PIPELINE_VERSION,
    })

if enrichment_refresh_rows:
    df_enrichment_refresh_new = pd.DataFrame(
        enrichment_refresh_rows
    )

    if not df_enrichment_refresh_state.empty:
        refreshed = safe_int_set(
            df_enrichment_refresh_new[
                "aaAthleteId"
            ]
        )

        df_enrichment_refresh_state = (
            df_enrichment_refresh_state[
                ~pd.to_numeric(
                    df_enrichment_refresh_state[
                        "aaAthleteId"
                    ],
                    errors="coerce"
                )
                .isin(
                    refreshed
                )
            ]
            .copy()
        )

    df_enrichment_refresh_state = pd.concat(
        [
            df_enrichment_refresh_state,
            df_enrichment_refresh_new
        ],
        ignore_index=True,
        sort=False
    )


df_enrichment_status = pd.DataFrame(
    [
        {
            "run_id":
                RUN_ID,

            "aaAthleteId":
                int(
                    athlete_id
                ),

            "status":
                "OK",

            "source":
                "api_incremental",

            "fetched_at_utc":
                enrichment_fetch_meta.get(
                    athlete_id
                ),

            "pipeline_version":
                PIPELINE_VERSION,
        }
        for athlete_id
        in sorted(
            refreshed_id_set
        )
    ]
)


# Update athlete-master identity fields from fresh profile data.
if not fresh_enrichment_tables[
    "profiles_enriched"
].empty:
    fresh_profiles = (
        fresh_enrichment_tables[
            "profiles_enriched"
        ]
        .copy()
    )

    update_columns = [
        "givenName",
        "familyName",
        "birthDate",
        "countryCode",
        "countryFullName",
        "gender",
    ]

    master = (
        df_all_sgp_athletes
        .copy()
    )

    master[
        "aaAthleteId"
    ] = pd.to_numeric(
        master[
            "aaAthleteId"
        ],
        errors="coerce"
    ).astype("Int64")

    fresh_profiles[
        "aaAthleteId"
    ] = pd.to_numeric(
        fresh_profiles[
            "aaAthleteId"
        ],
        errors="coerce"
    ).astype("Int64")

    fresh_indexed = (
        fresh_profiles
        .drop_duplicates(
            subset=[
                "aaAthleteId"
            ],
            keep="last"
        )
        .set_index(
            "aaAthleteId"
        )
    )

    for athlete_id in (
        fresh_indexed.index
    ):
        mask = master[
            "aaAthleteId"
        ].eq(
            athlete_id
        )

        if not mask.any():
            continue

        for col in update_columns:
            if (
                col
                in fresh_indexed.columns
            ):
                value = fresh_indexed.at[
                    athlete_id,
                    col
                ]

                if not (
                    value is None
                    or (
                        not isinstance(
                            value,
                            (list, dict)
                        )
                        and pd.isna(
                            value
                        )
                    )
                ):
                    master.loc[
                        mask,
                        col
                    ] = value

    df_all_sgp_athletes = master


print(
    "Current enriched profiles:",
    len(
        df_profiles_enriched
    )
)

print(
    "Current PB rows:",
    len(
        df_personal_bests
    )
)

print(
    "Current season-best rows:",
    len(
        df_season_bests_2026
    )
)

Current enriched profiles: 219
Current PB rows: 658
Current season-best rows: 477


## 16. QA and season-best reconciliation

Blocking QA is run against the **current merged state**, not only the delta.

The known competition backend exception remains non-blocking if explicitly classified as a known source-system exception.

In [51]:
# 16A. Rebuild current statuses after recursive discovery

profile_ids = safe_int_set(
    df_all_sgp_athletes[
        "aaAthleteId"
    ]
)

result_ids = safe_int_set(
    df_results_2026.get(
        "aaAthleteId",
        pd.Series(dtype=object)
    )
)

active_ids = set(
    result_ids
)

enriched_ids = safe_int_set(
    df_profiles_enriched.get(
        "aaAthleteId",
        pd.Series(dtype=object)
    )
)

active_2026_ids = sorted(
    active_ids
)

result_counts = (
    df_results_2026[
        "aaAthleteId"
    ]
    .value_counts()
    if not df_results_2026.empty
    else pd.Series(dtype="int64")
)

df_result_status = pd.DataFrame({
    "aaAthleteId":
        sorted(
            profile_ids
        )
})

df_result_status[
    "resultRows"
] = (
    df_result_status[
        "aaAthleteId"
    ]
    .map(
        result_counts
    )
    .fillna(0)
    .astype(int)
)

df_result_status[
    "active"
] = (
    df_result_status[
        "resultRows"
    ] > 0
)

df_result_status[
    "run_id"
] = RUN_ID

df_result_status[
    "pipeline_version"
] = PIPELINE_VERSION


qa = {
    "profiles":
        len(profile_ids),

    "active":
        len(active_ids),

    "resultAthletes":
        len(result_ids),

    "enrichedActive":
        len(
            active_ids
            & enriched_ids
        ),

    "activeMissingResults":
        sorted(
            active_ids
            - result_ids
        ),

    "activeMissingEnrichment":
        sorted(
            active_ids
            - enriched_ids
        ),

    "resultAthletesNotActive":
        sorted(
            result_ids
            - active_ids
        ),

    "activeMissingProfile":
        sorted(
            active_ids
            - profile_ids
        ),

    "resultFetchErrors":
        len(
            df_result_errors
        ),

    "enrichmentErrors":
        len(
            df_enrichment_errors
        ),

    "newProfiles":
        len(
            new_profile_ids
        ),

    "resultChangeEvents":
        len(
            df_result_changes
        ),

    "newCompetitionsProcessed":
        len(
            competition_refresh_rows
        ),
}


qa_display = {
    k:
        len(v)
        if isinstance(
            v,
            list
        )
        else v

    for k, v in qa.items()
}

display(
    pd.DataFrame(
        [qa_display]
    )
)

assert not qa[
    "activeMissingResults"
]

assert not qa[
    "activeMissingEnrichment"
]

assert not qa[
    "resultAthletesNotActive"
]

assert not qa[
    "activeMissingProfile"
]

assert qa[
    "resultFetchErrors"
] == 0

assert qa[
    "enrichmentErrors"
] == 0

print(
    "Blocking current-state population QA passed."
)

,profiles,active,resultAthletes,enrichedActive,activeMissingResults,activeMissingEnrichment,resultAthletesNotActive,activeMissingProfile,resultFetchErrors,enrichmentErrors,newProfiles,resultChangeEvents,newCompetitionsProcessed
0,398,219,219,219,0,0,0,0,0,0,0,4,0


Blocking current-state population QA passed.


In [52]:
def comparison_series(frame: pd.DataFrame, column: str):
    if column in frame.columns:
        return frame[column]
    return pd.Series(pd.NA, index=frame.index, dtype="object")


def reconcile_season_bests(df_results, df_sb):
    if df_results.empty or df_sb.empty:
        return pd.DataFrame()

    r = df_results.copy()
    s = df_sb.copy()

    for frame in [r, s]:
        frame["aaAthleteId"] = pd.to_numeric(
            comparison_series(frame, "aaAthleteId"),
            errors="coerce"
        ).astype("Int64")

        frame["competitionId_cmp"] = pd.to_numeric(
            comparison_series(frame, "competitionId"),
            errors="coerce"
        ).astype("Int64")

        frame["date_cmp"] = (
            comparison_series(frame, "date")
            .astype(str)
            .str.strip()
        )

        frame["mark_cmp"] = comparison_series(frame, "mark").map(
            normalize_mark
        )

    return s.merge(
        r[
            [
                "aaAthleteId",
                "date_cmp",
                "competitionId_cmp",
                "mark_cmp"
            ]
        ].drop_duplicates(),
        on=[
            "aaAthleteId",
            "date_cmp",
            "competitionId_cmp",
            "mark_cmp"
        ],
        how="left",
        indicator=True
    )


df_sb_reconciliation = reconcile_season_bests(
    df_results_2026,
    df_season_bests_2026
)

if not df_sb_reconciliation.empty:
    rate = df_sb_reconciliation["_merge"].eq("both").mean()
    print("SB reconciliation:", f"{rate:.2%}")
    assert rate == 1.0

SB reconciliation: 100.00%


## 17. Save only changed current tables, write immutable versions and append audit history

Parquet does not support safe in-place row append to a single monolithic file. Therefore the pipeline uses two layers:

- **append-only delta files** under `deltas/<RUN_ID>/`;
- an atomically replaced **current canonical snapshot** at the root.

Every current table whose content hash changed receives an immutable snapshot under `versions/<RUN_ID>/tables/`.

Every table is audited with before/after row count, schema hash and content SHA256.

In [53]:
# 17A. Assemble current tables and run-level deltas

# Search outputs may be loaded by the optional search stage.
if "df_search_audit" not in globals():
    df_search_audit = baseline_tables[
        "search_index_audit"
    ].copy()

if "df_search_errors" not in globals():
    df_search_errors = pd.DataFrame()


# Append current run's status/error records to current audit-like tables.
df_enrichment_status_current = pd.concat(
    [
        baseline_tables[
            "enrichment_status"
        ],
        df_enrichment_status
    ],
    ignore_index=True,
    sort=False
)

df_enrichment_errors_current = pd.concat(
    [
        baseline_tables[
            "enrichment_errors"
        ],
        df_enrichment_errors
    ],
    ignore_index=True,
    sort=False
)

df_result_errors_current = pd.concat(
    [
        baseline_tables[
            "result_errors"
        ],
        df_result_errors
    ],
    ignore_index=True,
    sort=False
)



# Add canonical athleteName to every athlete-keyed output.
df_all_sgp_athletes = ensure_master_athlete_name(
    df_all_sgp_athletes
)

_frames_with_athletes = {
    "results": df_results_2026,
    "profiles_enriched": df_profiles_enriched,
    "personal_bests": df_personal_bests,
    "season_bests": df_season_bests_2026,
    "progression": df_progression,
    "rankings_current": df_rankings_current,
    "rankings_best": df_rankings_best,
    "honours": df_honours,
    "competition_candidate_audit": df_competition_candidate_audit,
    "result_status": df_result_status,
    "result_errors": df_result_errors_current,
    "enrichment_status": df_enrichment_status_current,
    "enrichment_errors": df_enrichment_errors_current,
    "athlete_refresh_state": df_result_refresh_state,
    "enrichment_refresh_state": df_enrichment_refresh_state,
    "sb_reconciliation": df_sb_reconciliation,
    "result_changes": df_result_changes,
    "new_competition_candidate_audit": df_new_candidate_audit,
}

for _key, _frame in _frames_with_athletes.items():
    _frames_with_athletes[_key] = attach_athlete_name(
        _frame,
        df_all_sgp_athletes
    )

df_results_2026 = _frames_with_athletes["results"]
df_profiles_enriched = _frames_with_athletes["profiles_enriched"]
df_personal_bests = _frames_with_athletes["personal_bests"]
df_season_bests_2026 = _frames_with_athletes["season_bests"]
df_progression = _frames_with_athletes["progression"]
df_rankings_current = _frames_with_athletes["rankings_current"]
df_rankings_best = _frames_with_athletes["rankings_best"]
df_honours = _frames_with_athletes["honours"]
df_competition_candidate_audit = _frames_with_athletes["competition_candidate_audit"]
df_result_status = _frames_with_athletes["result_status"]
df_result_errors_current = _frames_with_athletes["result_errors"]
df_enrichment_status_current = _frames_with_athletes["enrichment_status"]
df_enrichment_errors_current = _frames_with_athletes["enrichment_errors"]
df_result_refresh_state = _frames_with_athletes["athlete_refresh_state"]
df_enrichment_refresh_state = _frames_with_athletes["enrichment_refresh_state"]
df_sb_reconciliation = _frames_with_athletes["sb_reconciliation"]
df_result_changes = _frames_with_athletes["result_changes"]
df_new_candidate_audit = _frames_with_athletes["new_competition_candidate_audit"]


# Current competition exceptions: keep historical exception records,
# but avoid exact duplicate rows.
if not df_competition_exceptions.empty:
    df_competition_exceptions = (
        df_competition_exceptions
        .drop_duplicates()
        .reset_index(drop=True)
    )


tables = {
    "athlete_master":
        df_all_sgp_athletes,

    f"results_{RESULTS_YEAR}":
        df_results_2026,

    "profiles_enriched":
        df_profiles_enriched,

    "personal_bests":
        df_personal_bests,

    f"season_bests_{RESULTS_YEAR}":
        df_season_bests_2026,

    "progression":
        df_progression,

    "rankings_current":
        df_rankings_current,

    "rankings_best":
        df_rankings_best,

    "honours":
        df_honours,

    "competition_discovery":
        df_competition_sgp,

    "competition_candidate_audit":
        df_competition_candidate_audit,

    "competition_exceptions":
        df_competition_exceptions,

    "result_status":
        df_result_status,

    "result_errors":
        df_result_errors_current,

    "enrichment_status":
        df_enrichment_status_current,

    "enrichment_errors":
        df_enrichment_errors_current,

    "search_index_audit":
        df_search_audit,

    "search_index_errors":
        df_search_errors,

    "sb_reconciliation":
        df_sb_reconciliation,

    "athlete_refresh_state":
        df_result_refresh_state,

    "enrichment_refresh_state":
        df_enrichment_refresh_state,

    "competition_refresh_state":
        df_competition_refresh_state,
}


# Run-specific delta evidence.
delta_tables = {
    "result_changes":
        df_result_changes,

    "new_competition_rows":
        df_new_competition_rows,

    "new_competition_exceptions":
        df_new_competition_exceptions,

    "new_competition_candidate_audit":
        df_new_candidate_audit,

    "result_refresh_errors":
        df_result_errors,

    "enrichment_refresh_errors":
        df_enrichment_errors,
}


for name, df in (
    delta_tables.items()
):
    if (
        df is not None
        and not df.empty
    ):
        save_parquet_atomic(
            json_safe_export_copy(
                df
            ),
            RUN_DELTA_DIR
            / f"{name}.parquet"
        )


print(
    "Run delta directory:",
    RUN_DELTA_DIR.resolve()
)

Run delta directory: /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_SGP_2026/deltas/20260927T071746Z_b5c61458


In [54]:
# 17B. Save changed tables + immutable version snapshots + table audit

table_change_rows = []
save_report_rows = []

save_started = time.monotonic()

for i, (name, df) in enumerate(
    tables.items(),
    start=1
):
    before_hash = baseline_table_hashes.get(
        name
    )

    after_hash = dataframe_sha256(
        df
    )

    before_rows = baseline_row_counts.get(
        name,
        0
    )

    after_rows = len(
        df
    )

    changed = (
        before_hash
        != after_hash
    )

    root_parquet = (
        OUTPUT_DIR
        / f"{name}.parquet"
    )

    root_csv = (
        OUTPUT_DIR
        / f"{name}.csv"
    )

    version_path = None

    try:
        if changed:
            export_df = json_safe_export_copy(
                df
            )

            # Atomically materialize the latest current state.
            save_parquet_atomic(
                export_df,
                root_parquet
            )

            save_csv_atomic(
                export_df,
                root_csv
            )

            # Immutable table snapshot for this data version.
            version_path = (
                RUN_VERSION_DIR
                / "tables"
                / f"{name}.parquet"
            )

            save_parquet_atomic(
                export_df,
                version_path
            )

        save_report_rows.append({
            "run_id":
                RUN_ID,

            "table":
                name,

            "status":
                "OK",

            "changed":
                changed,

            "rows_before":
                before_rows,

            "rows_after":
                after_rows,

            "before_sha256":
                before_hash,

            "after_sha256":
                after_hash,

            "schema_sha256":
                table_schema_hash(
                    df
                ),

            "current_path":
                str(
                    root_parquet
                ),

            "version_path":
                str(
                    version_path
                )
                if version_path
                else None,
        })

    except Exception as exc:
        save_report_rows.append({
            "run_id":
                RUN_ID,

            "table":
                name,

            "status":
                "ERROR",

            "changed":
                changed,

            "rows_before":
                before_rows,

            "rows_after":
                after_rows,

            "error":
                repr(exc),
        })

    table_change_rows.append({
        "run_id":
            RUN_ID,

        "data_version":
            DATA_VERSION,

        "pipeline_version":
            PIPELINE_VERSION,

        "table":
            name,

        "change_type":
            "CHANGED"
            if changed
            else "NO_CHANGE",

        "rows_before":
            before_rows,

        "rows_after":
            after_rows,

        "row_delta":
            after_rows
            - before_rows,

        "before_sha256":
            before_hash,

        "after_sha256":
            after_hash,

        "schema_sha256":
            table_schema_hash(
                df
            ),

        "version_path":
            str(
                version_path
            )
            if version_path
            else None,

        "recorded_at_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    })

    print_progress(
        "Saving/versioning tables",
        i,
        len(tables),
        save_started,
        extra=(
            f"{name} "
            f"| {'CHANGED' if changed else 'no change'}"
        )
    )


df_save_report = pd.DataFrame(
    save_report_rows
)

df_table_change_run = pd.DataFrame(
    table_change_rows
)

display(
    df_save_report
)

failed_saves = df_save_report[
    df_save_report[
        "status"
    ].ne("OK")
]

assert failed_saves.empty, (
    "One or more production tables failed to save. "
    "Inspect df_save_report."
)


# Append cumulative table audit.
df_table_change_history = append_audit_parquet(
    AUDIT_DIR
    / "table_change_history.parquet",
    df_table_change_run,
)


# Append cumulative row-level result audit.
if not df_result_changes.empty:
    df_result_change_history = append_audit_parquet(
        AUDIT_DIR
        / "result_change_history.parquet",
        df_result_changes,
    )
else:
    result_history_path = (
        AUDIT_DIR
        / "result_change_history.parquet"
    )

    df_result_change_history = (
        pd.read_parquet(
            result_history_path
        )
        if result_history_path.exists()
        else pd.DataFrame()
    )

Saving/versioning tables: 1/22 (4.5%) | elapsed=0s | rate=54.04/s | ETA=0s | athlete_master | no change
Saving/versioning tables: 2/22 (9.1%) | elapsed=0s | rate=3.53/s | ETA=5s | results_2026 | CHANGED
Saving/versioning tables: 3/22 (13.6%) | elapsed=1s | rate=2.60/s | ETA=7s | profiles_enriched | CHANGED
Saving/versioning tables: 4/22 (18.2%) | elapsed=1s | rate=3.41/s | ETA=5s | personal_bests | no change
Saving/versioning tables: 5/22 (22.7%) | elapsed=1s | rate=3.67/s | ETA=4s | season_bests_2026 | CHANGED
Saving/versioning tables: 6/22 (27.3%) | elapsed=1s | rate=3.71/s | ETA=4s | progression | CHANGED
Saving/versioning tables: 7/22 (31.8%) | elapsed=1s | rate=4.03/s | ETA=3s | rankings_current | CHANGED
Saving/versioning tables: 8/22 (36.4%) | elapsed=1s | rate=4.59/s | ETA=3s | rankings_best | no change
Saving/versioning tables: 9/22 (40.9%) | elapsed=1s | rate=5.15/s | ETA=2s | honours | no change
Saving/versioning tables: 10/22 (45.5%) | elapsed=1s | rate=5.68/s | ETA=2s | co

,run_id,table,status,changed,rows_before,rows_after,before_sha256,after_sha256,schema_sha256,current_path,version_path
0,20260927T071746Z_b5c61458,athlete_master,OK,False,398,398,db2b968fc2849a8bcfd5c16f56bc9ef411a85c6c953e59...,db2b968fc2849a8bcfd5c16f56bc9ef411a85c6c953e59...,73eee8a396bb5d24e02e9fea39eef3ef26883dce4a5bd9...,world_athletics_SGP_2026/athlete_master.parquet,None
1,20260927T071746Z_b5c61458,results_2026,OK,True,1214,1218,f0d9f0bd6e2700d5258296af2b098d933d817a55654e11...,238439c8e65dfb6707d1f7bc4e8d2537ada7e0a80d342d...,bcc4a6ed57a217cc7d30016c98fb358b63c384b3545ea7...,world_athletics_SGP_2026/results_2026.parquet,world_athletics_SGP_2026/versions/20260927T071...
2,20260927T071746Z_b5c61458,profiles_enriched,OK,True,219,219,1bb73413592b12878893690eeb2afe394259bb1f290de9...,5c065a7cca1d13778890bb6bdaf5aa3c29f166a710d810...,4318b8b0482df77a84be5760e7e8bc1fe95feb08048df7...,world_athletics_SGP_2026/profiles_enriched.par...,world_athletics_SGP_2026/versions/20260927T071...
3,20260927T071746Z_b5c61458,personal_bests,OK,False,658,658,cd5e6c96ed62b41c6d583f86ee7222785e35f41c857758...,cd5e6c96ed62b41c6d583f86ee7222785e35f41c857758...,96ff37c1f7fe510964ec0c3e79a11c2b0868f8afaac867...,world_athletics_SGP_2026/personal_bests.parquet,None
4,20260927T071746Z_b5c61458,season_bests_2026,OK,True,477,477,1bff2454b0cefd7b35b898ca9a201906ec24b0fc38a192...,a172cf35d0e12adf745f76f7ae07a3a459b9ba1fc3da43...,96ff37c1f7fe510964ec0c3e79a11c2b0868f8afaac867...,world_athletics_SGP_2026/season_bests_2026.par...,world_athletics_SGP_2026/versions/20260927T071...
5,20260927T071746Z_b5c61458,progression,OK,True,1359,1359,594ccd65d10a1cc1065d8b85ea93b145152bad11b6c965...,217896dc567f1d244d8830f79d6ae5678fa02580c650c7...,8dd8255e40921e510e44817e2a039c335a57bdb5fbdb59...,world_athletics_SGP_2026/progression.parquet,world_athletics_SGP_2026/versions/20260927T071...
6,20260927T071746Z_b5c61458,rankings_current,OK,True,106,106,697c1a667c428475e75cbe579e9f167d6344641e0bc979...,ddede4747e6041396615f557d12abe7a231b04590fd604...,014c664093e5ee3d429c49297f5e5e9f3fee38c94176a4...,world_athletics_SGP_2026/rankings_current.parquet,world_athletics_SGP_2026/versions/20260927T071...
7,20260927T071746Z_b5c61458,rankings_best,OK,False,148,148,7429cc6f91fbfe097ce971b07a78be65a32425af830f46...,7429cc6f91fbfe097ce971b07a78be65a32425af830f46...,e0ee4cb7f2f060cb27799f2b40b21dc46e7714be60ef90...,world_athletics_SGP_2026/rankings_best.parquet,None
8,20260927T071746Z_b5c61458,honours,OK,False,74,74,52aa15a8b94a4fd3f835cc01d118204f4ec89bd79f9b6e...,52aa15a8b94a4fd3f835cc01d118204f4ec89bd79f9b6e...,bf99cb864f70652b58ba3262907b5b39e3574e1306565e...,world_athletics_SGP_2026/honours.parquet,None
9,20260927T071746Z_b5c61458,competition_discovery,OK,False,451,451,4789987d2d92c27c06b2968f93f545cf5e2b19482b051e...,4789987d2d92c27c06b2968f93f545cf5e2b19482b051e...,f449c9d51884702faea71d63d7df16dd8786f1db8faffb...,world_athletics_SGP_2026/competition_discovery...,None


In [55]:
# 17C. Code-version history, run manifest, run history and latest pointer

RUN_FINISHED_AT = datetime.now(
    timezone.utc
)

changed_tables = (
    df_table_change_run.loc[
        df_table_change_run[
            "change_type"
        ].eq(
            "CHANGED"
        ),
        "table"
    ]
    .tolist()
)


# Code-version history is append-only only when code hash/version combination changes.
code_history_path = (
    AUDIT_DIR
    / "code_version_history.parquet"
)

df_code_history = (
    pd.read_parquet(
        code_history_path
    )
    if code_history_path.exists()
    else pd.DataFrame()
)

code_combo_exists = False

if (
    not df_code_history.empty
    and {
        "pipeline_version",
        "notebook_code_hash"
    }.issubset(
        df_code_history.columns
    )
):
    code_combo_exists = (
        (
            df_code_history[
                "pipeline_version"
            ].astype(str)
            == PIPELINE_VERSION
        )
        & (
            df_code_history[
                "notebook_code_hash"
            ].astype(str)
            == str(
                NOTEBOOK_CODE_HASH
            )
        )
    ).any()


if not code_combo_exists:
    code_row = pd.DataFrame(
        [
            {
                "pipeline_version":
                    PIPELINE_VERSION,

                "notebook_code_hash":
                    NOTEBOOK_CODE_HASH,

                "notebook_path":
                    str(
                        notebook_path
                    ),

                "change_log":
                    json.dumps(
                        CODE_CHANGELOG,
                        ensure_ascii=False
                    ),

                "first_seen_run_id":
                    RUN_ID,

                "recorded_at_utc":
                    RUN_FINISHED_AT.isoformat(),
            }
        ]
    )

    df_code_history = append_audit_parquet(
        code_history_path,
        code_row,
    )


# High-watermark after run.
result_dates = (
    parse_wa_dates(
        df_results_2026[
            "date"
        ]
    )
    if (
        not df_results_2026.empty
        and "date"
        in df_results_2026.columns
    )
    else pd.Series(
        dtype="datetime64[ns, UTC]"
    )
)

result_high_watermark = (
    result_dates.max()
    if not result_dates.empty
    else pd.NaT
)


result_change_counts = (
    df_result_changes[
        "change_type"
    ]
    .value_counts()
    .to_dict()
    if not df_result_changes.empty
    else {}
)


manifest = {
    "pipelineName":
        PIPELINE_NAME,

    "pipelineVersion":
        PIPELINE_VERSION,

    "notebookCodeHash":
        NOTEBOOK_CODE_HASH,

    "runId":
        RUN_ID,

    "dataVersion":
        DATA_VERSION,

    "countryCode":
        COUNTRY_CODE,

    "resultsYear":
        RESULTS_YEAR,

    "runStartedAt":
        RUN_STARTED_AT.isoformat(),

    "runFinishedAt":
        RUN_FINISHED_AT.isoformat(),

    "durationSeconds":
        (
            RUN_FINISHED_AT
            - RUN_STARTED_AT
        ).total_seconds(),

    "graphqlEndpoint":
        wa_api["url"],

    "mode":
        "incremental",

    "settings": {
        "activeResultRefreshHours":
            ACTIVE_RESULT_REFRESH_HOURS,

        "inactiveResultRefreshDays":
            INACTIVE_RESULT_REFRESH_DAYS,

        "resultReconciliationLookbackDays":
            RESULT_RECONCILIATION_LOOKBACK_DAYS,

        "forceFullYearReconciliation":
            FORCE_FULL_YEAR_RECONCILIATION,

        "enrichmentRefreshDays":
            ENRICHMENT_REFRESH_DAYS,

        "runSearchIndexRefresh":
            RUN_SEARCH_INDEX_REFRESH,

        "retryKnownCompetitionExceptions":
            RETRY_KNOWN_COMPETITION_EXCEPTIONS,

        "apiMaxConcurrency":
            API_MAX_CONCURRENCY,
    },

    "counts": {
        "profilesCurrent":
            len(
                profile_ids
            ),

        "activeAthletesCurrent":
            len(
                active_ids
            ),

        "resultRowsCurrent":
            len(
                df_results_2026
            ),

        "enrichedProfilesCurrent":
            len(
                active_ids
                & enriched_ids
            ),

        "athletesQueriedForResults":
            len(
                result_pairs
            ),

        "athletesSkippedResultRefresh":
            len(
                skipped_profile_ids
            ),

        "athletesWithResultChanges":
            len(
                changed_result_athlete_ids
            ),

        "newProfiles":
            len(
                new_profile_ids
            ),

        "newCompetitionRows":
            len(
                df_new_competition_rows
            ),

        "enrichmentAthletesRefreshed":
            len(
                refreshed_id_set
            ),

        "changedTables":
            len(
                changed_tables
            ),
    },

    "resultChangeCounts":
        result_change_counts,

    "resultHighWatermark":
        (
            result_high_watermark.isoformat()
            if pd.notna(
                result_high_watermark
            )
            else None
        ),

    "changedTables":
        changed_tables,

    "qa":
        qa,

    "apiStats":
        _api_stats,

    "knownSourceSystemExceptions":
        sorted(
            safe_int_set(
                df_competition_exceptions.get(
                    "competitionId",
                    pd.Series(
                        dtype=object
                    )
                )
            )
        ),

    "paths": {
        "versionDirectory":
            str(
                RUN_VERSION_DIR
            ),

        "deltaDirectory":
            str(
                RUN_DELTA_DIR
            ),

        "auditDirectory":
            str(
                AUDIT_DIR
            ),
    },
}


# Immutable manifest for this version.
write_json_atomic(
    manifest,
    RUN_VERSION_DIR
    / "manifest.json"
)

# Latest manifest for convenient access.
write_json_atomic(
    manifest,
    OUTPUT_DIR
    / "run_manifest.json"
)

# Latest pointer.
write_json_atomic(
    {
        "runId":
            RUN_ID,

        "dataVersion":
            DATA_VERSION,

        "pipelineVersion":
            PIPELINE_VERSION,

        "manifest":
            str(
                RUN_VERSION_DIR
                / "manifest.json"
            ),

        "updatedAt":
            RUN_FINISHED_AT.isoformat(),
    },
    OUTPUT_DIR
    / "LATEST.json"
)


# Run history.
run_history_row = pd.DataFrame(
    [
        {
            "run_id":
                RUN_ID,

            "data_version":
                DATA_VERSION,

            "pipeline_version":
                PIPELINE_VERSION,

            "notebook_code_hash":
                NOTEBOOK_CODE_HASH,

            "started_at_utc":
                RUN_STARTED_AT.isoformat(),

            "finished_at_utc":
                RUN_FINISHED_AT.isoformat(),

            "duration_seconds":
                manifest[
                    "durationSeconds"
                ],

            "profiles_current":
                len(
                    profile_ids
                ),

            "active_athletes_current":
                len(
                    active_ids
                ),

            "result_rows_current":
                len(
                    df_results_2026
                ),

            "result_change_events":
                len(
                    df_result_changes
                ),

            "changed_tables":
                len(
                    changed_tables
                ),

            "api_requests":
                _api_stats.get(
                    "requests",
                    0
                ),

            "api_retries":
                _api_stats.get(
                    "retries",
                    0
                ),

            "status":
                "SUCCESS",
        }
    ]
)

df_run_history = append_audit_parquet(
    AUDIT_DIR
    / "run_history.parquet",
    run_history_row,
)


# Save this run's save report.
save_parquet_atomic(
    df_save_report,
    RUN_VERSION_DIR
    / "save_report.parquet"
)


# Persist state only after all saves and audit writes succeeded.
new_pipeline_state = {
    "pipelineName":
        PIPELINE_NAME,

    "pipelineVersion":
        PIPELINE_VERSION,

    "notebookCodeHash":
        NOTEBOOK_CODE_HASH,

    "lastSuccessfulRunId":
        RUN_ID,

    "lastSuccessfulDataVersion":
        DATA_VERSION,

    "lastSuccessfulRunAt":
        RUN_FINISHED_AT.isoformat(),

    "lastResultHighWatermark":
        manifest[
            "resultHighWatermark"
        ],
}

write_json_atomic(
    new_pipeline_state,
    PIPELINE_STATE_PATH
)


print()
print(
    "Incremental production run COMPLETE."
)

print(
    "Run ID:",
    RUN_ID
)

print(
    "Changed tables:",
    changed_tables
)

print(
    "Result changes:",
    result_change_counts
)

print(
    "Current result high-watermark:",
    manifest[
        "resultHighWatermark"
    ]
)

print(
    "Version manifest:",
    (
        RUN_VERSION_DIR
        / "manifest.json"
    ).resolve()
)


Incremental production run COMPLETE.
Run ID: 20260927T071746Z_b5c61458
Changed tables: ['results_2026', 'profiles_enriched', 'season_bests_2026', 'progression', 'rankings_current', 'result_status', 'enrichment_status', 'sb_reconciliation', 'athlete_refresh_state', 'enrichment_refresh_state']
Result changes: {'INSERT': 4}
Current result high-watermark: 2026-09-26T00:00:00+00:00
Version manifest: /Users/veesheenyuen/Documents/code/veeyuen/SAA/world_athletics_SGP_2026/versions/20260927T071746Z_b5c61458/manifest.json


## 18. Verify current outputs and audit/version history

In [56]:
# 18. Final verification / operator summary

core_checks = {
    "athlete_master.parquet":
        OUTPUT_DIR
        / "athlete_master.parquet",

    f"results_{RESULTS_YEAR}.parquet":
        OUTPUT_DIR
        / f"results_{RESULTS_YEAR}.parquet",

    "profiles_enriched.parquet":
        OUTPUT_DIR
        / "profiles_enriched.parquet",

    "personal_bests.parquet":
        OUTPUT_DIR
        / "personal_bests.parquet",

    f"season_bests_{RESULTS_YEAR}.parquet":
        OUTPUT_DIR
        / f"season_bests_{RESULTS_YEAR}.parquet",

    "run_manifest.json":
        OUTPUT_DIR
        / "run_manifest.json",

    "LATEST.json":
        OUTPUT_DIR
        / "LATEST.json",

    "audit/run_history.parquet":
        AUDIT_DIR
        / "run_history.parquet",

    "audit/table_change_history.parquet":
        AUDIT_DIR
        / "table_change_history.parquet",

    "audit/result_change_history.parquet":
        AUDIT_DIR
        / "result_change_history.parquet",
}


for label, path in (
    core_checks.items()
):
    print(
        f"{label:<42}",
        "FOUND"
        if path.exists()
        else "MISSING"
    )


print()
print(
    "Current profiles:",
    len(
        df_all_sgp_athletes
    )
)

print(
    "Current active/result athletes:",
    df_results_2026[
        "aaAthleteId"
    ].nunique()
    if not df_results_2026.empty
    else 0
)

print(
    "Current result rows:",
    len(
        df_results_2026
    )
)

print(
    "Result changes this run:",
    len(
        df_result_changes
    )
)

print(
    "Tables changed this run:",
    len(
        changed_tables
    )
)


display(
    df_table_change_run[
        [
            "table",
            "change_type",
            "rows_before",
            "rows_after",
            "before_sha256",
            "after_sha256",
            "version_path",
        ]
    ]
)


if not df_result_changes.empty:
    display(
        df_result_changes
        .sort_values(
            [
                "aaAthleteId",
                "date"
            ],
            na_position="last"
        )
    )

# Athlete-name coverage across all athlete-keyed current outputs.
_name_coverage = []

for _table_name, _frame in tables.items():
    if (
        _frame is not None
        and not _frame.empty
        and "aaAthleteId" in _frame.columns
    ):
        if ATHLETE_NAME_COLUMN in _frame.columns:
            _missing = (
                _frame[ATHLETE_NAME_COLUMN]
                .astype("string")
                .fillna("")
                .str.strip()
                .eq("")
                .sum()
            )
        else:
            _missing = len(_frame)

        _name_coverage.append({
            "table": _table_name,
            "rows": len(_frame),
            "missingAthleteName": int(_missing),
        })

df_athlete_name_coverage = pd.DataFrame(
    _name_coverage
)

print()
print("Athlete-name coverage:")
display(df_athlete_name_coverage)


athlete_master.parquet                     FOUND
results_2026.parquet                       FOUND
profiles_enriched.parquet                  FOUND
personal_bests.parquet                     FOUND
season_bests_2026.parquet                  FOUND
run_manifest.json                          FOUND
LATEST.json                                FOUND
audit/run_history.parquet                  FOUND
audit/table_change_history.parquet         FOUND
audit/result_change_history.parquet        FOUND

Current profiles: 398
Current active/result athletes: 219
Current result rows: 1218
Result changes this run: 4
Tables changed this run: 10


,table,change_type,rows_before,rows_after,before_sha256,after_sha256,version_path
0,athlete_master,NO_CHANGE,398,398,db2b968fc2849a8bcfd5c16f56bc9ef411a85c6c953e59...,db2b968fc2849a8bcfd5c16f56bc9ef411a85c6c953e59...,None
1,results_2026,CHANGED,1214,1218,f0d9f0bd6e2700d5258296af2b098d933d817a55654e11...,238439c8e65dfb6707d1f7bc4e8d2537ada7e0a80d342d...,world_athletics_SGP_2026/versions/20260927T071...
2,profiles_enriched,CHANGED,219,219,1bb73413592b12878893690eeb2afe394259bb1f290de9...,5c065a7cca1d13778890bb6bdaf5aa3c29f166a710d810...,world_athletics_SGP_2026/versions/20260927T071...
3,personal_bests,NO_CHANGE,658,658,cd5e6c96ed62b41c6d583f86ee7222785e35f41c857758...,cd5e6c96ed62b41c6d583f86ee7222785e35f41c857758...,None
4,season_bests_2026,CHANGED,477,477,1bff2454b0cefd7b35b898ca9a201906ec24b0fc38a192...,a172cf35d0e12adf745f76f7ae07a3a459b9ba1fc3da43...,world_athletics_SGP_2026/versions/20260927T071...
5,progression,CHANGED,1359,1359,594ccd65d10a1cc1065d8b85ea93b145152bad11b6c965...,217896dc567f1d244d8830f79d6ae5678fa02580c650c7...,world_athletics_SGP_2026/versions/20260927T071...
6,rankings_current,CHANGED,106,106,697c1a667c428475e75cbe579e9f167d6344641e0bc979...,ddede4747e6041396615f557d12abe7a231b04590fd604...,world_athletics_SGP_2026/versions/20260927T071...
7,rankings_best,NO_CHANGE,148,148,7429cc6f91fbfe097ce971b07a78be65a32425af830f46...,7429cc6f91fbfe097ce971b07a78be65a32425af830f46...,None
8,honours,NO_CHANGE,74,74,52aa15a8b94a4fd3f835cc01d118204f4ec89bd79f9b6e...,52aa15a8b94a4fd3f835cc01d118204f4ec89bd79f9b6e...,None
9,competition_discovery,NO_CHANGE,451,451,4789987d2d92c27c06b2968f93f545cf5e2b19482b051e...,4789987d2d92c27c06b2968f93f545cf5e2b19482b051e...,None


,run_id,pipeline_version,change_type,aaAthleteId,athleteName,date,competitionId,discipline,result_identity_key,old_result_row_hash,new_result_row_hash,source_fetched_at_utc,changed_at_utc
0,20260927T071746Z_b5c61458,3.1.0,INSERT,14445804,Chen Xiang ANG,26 SEP 2026,7176091,110 Metres Hurdles,69d17befb49efab0e22a3eae89eccc88e4175f62496964...,None,a3871915d5fde67d39bd9f18e54f13aa7bfe6d82d408ca...,2026-09-27T07:18:28.116470+00:00,2026-09-27T07:18:38.585918+00:00
1,20260927T071746Z_b5c61458,3.1.0,INSERT,14472153,Veronica Shanti PEREIRA,26 SEP 2026,7176091,200 Metres,0828ca05dd0c511fde9ce51fc425d3fedc43b3a6f52e3a...,None,38ea71006cf79dc5459cb1825ec2f9f7dc6255fe2d47e4...,2026-09-27T07:18:28.116882+00:00,2026-09-27T07:18:38.824305+00:00
2,20260927T071746Z_b5c61458,3.1.0,INSERT,14845615,Elizabeth-Ann Shee Ru TAN,26 SEP 2026,7176091,200 Metres,855b81b23cf59966c599ab9eccb94edea4a4f7f66c5f55...,None,d8fb73ee89a55f09bff46ca55b1fde170c49f5227dc168...,2026-09-27T07:18:28.988008+00:00,2026-09-27T07:18:41.186018+00:00
3,20260927T071746Z_b5c61458,3.1.0,INSERT,14970362,Gabriel LEE,26 SEP 2026,7176091,Triple Jump,b5978842488f82b363504a28ec62faaa618880a0ced854...,None,c3f138aad729da1710839ef432a462f9b50f32eed0ab3f...,2026-09-27T07:18:29.434673+00:00,2026-09-27T07:18:41.894784+00:00



Athlete-name coverage:


,table,rows,missingAthleteName
0,athlete_master,398,0
1,results_2026,1218,0
2,profiles_enriched,219,0
3,personal_bests,658,0
4,season_bests_2026,477,0
5,progression,1359,0
6,rankings_current,106,0
7,rankings_best,148,0
8,honours,74,0
9,result_status,398,0


## 19. Close the persistent HTTP client

In [57]:
await _http_client.aclose()
print("HTTP client closed.")

HTTP client closed.
